# Data Cleaning

In [ ]:
%reset -f

Imports

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import gc
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit

Loading the datasheets

In [ ]:
DATA_RAW = Path("../data/raw")
studentInfo = pd.read_csv(DATA_RAW / "studentInfo.csv")
studentVle = pd.read_csv(DATA_RAW / "studentVle.csv")
assessments = pd.read_csv(DATA_RAW / "assessments.csv")
studentAssessment = pd.read_csv(DATA_RAW / "studentAssessment.csv")
courses = pd.read_csv(DATA_RAW / "courses.csv")
vle = pd.read_csv(DATA_RAW / "vle.csv")
studentReg = pd.read_csv(DATA_RAW / "studentRegistration.csv")

### Tables Overview

In [ ]:
datasets = {
    "studentInfo": studentInfo,
    "studentVle": studentVle,
    "assessments": assessments,
    "studentAssessment": studentAssessment,
    "courses": courses,
    "vle": vle,
    "studentRegistration": studentReg
}

for name, df in datasets.items():
    print(f"\n{'='*40}")
    print(f"{name}")
    print(f"{'='*40}")
    print("Shape:", df.shape)
    print("\nMissing values:\n", df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# =============================================================================
# SECTION 2 — Constants
# =============================================================================

KEYS        = ["id_student", "code_module", "code_presentation"]
COURSE_KEYS = ["code_module", "code_presentation"]
eps         = 1e-9

CONTENT_TYPES    = {"subpage","homepage","oucontent","resource","url","page",
                    "folder","glossary","htmlactivity","dualpane","repeatactivity"}
ASSESSMENT_TYPES = {"quiz","externalquiz","questionnaire"}
SOCIAL_TYPES     = {"forumng","ouwiki","oucollaborate","ouelluminate",
                    "dataplus","sharedsubpage"}
dim_keep         = {"content", "assessment", "social"}

In [ ]:
# =============================================================================
# SECTION 3 — Clean studentReg
# =============================================================================

studentReg["date_unregistration"] = pd.to_numeric(
    studentReg["date_unregistration"], errors="coerce")
studentReg["date_registration"] = pd.to_numeric(
    studentReg["date_registration"], errors="coerce")

studentReg = studentReg[
    ~(studentReg["date_unregistration"].notna() &
      (studentReg["date_unregistration"] < 0))
].copy()

studentReg["registration_missing_flag"] = studentReg["date_registration"].isna().astype(int)
studentReg["unregistered_flag"]         = studentReg["date_unregistration"].notna().astype(int)
studentReg["unregistration_week"]       = (
    (studentReg["date_unregistration"] // 7) + 1
).fillna(-1).astype(int)
studentReg["date_registration"]         = studentReg["date_registration"].fillna(0)
studentReg["late_registration_flag"]    = (studentReg["date_registration"] > 0).astype(int)



In [ ]:
# =============================================================================
# SECTION 4 — Courses + course_windows with FIXED quarter cutoffs
# =============================================================================

courses = courses.copy()
courses["module_presentation_length"] = pd.to_numeric(
    courses["module_presentation_length"], errors="coerce")
courses["course_weeks"] = np.ceil(
    courses["module_presentation_length"] / 7).astype("Int64")

non_stem_modules    = {"AAA", "BBB", "GGG"}
courses["course_type_stem"] = (
    ~courses["code_module"].astype(str).str.upper().str.strip().isin(non_stem_modules)
).astype(int)

print("Unique course week lengths:")
print(courses["course_weeks"].value_counts(dropna=False))

def assign_quarter_cutoffs(course_weeks):
    """
    Paper's fixed cutoffs:
      38-week: Q1=10, Q2=19, Q3=29, Q4=38
      34-week: Q1=9,  Q2=17, Q3=26, Q4=34
    All other lengths: proportional fallback.
    """
    if course_weeks == 38:
        return 10, 19, 29, 38
    elif course_weeks == 34:
        return 9, 17, 26, 34
    else:
        q1 = max(1, int(np.ceil(course_weeks * 0.25)))
        q2 = min(int(np.ceil(course_weeks * 0.50)), course_weeks)
        q3 = min(int(np.ceil(course_weeks * 0.75)), course_weeks)
        q4 = course_weeks
        return q1, q2, q3, q4

course_windows = courses[COURSE_KEYS + ["course_weeks"]].drop_duplicates().copy()
course_windows["course_weeks"] = course_windows["course_weeks"].fillna(1).clip(lower=1).astype(int)
cutoffs = course_windows["course_weeks"].apply(
    lambda cw: pd.Series(
        assign_quarter_cutoffs(cw), index=["q1_end","q2_end","q3_end","q4_end"]))
course_windows = pd.concat([course_windows, cutoffs], axis=1)
course_windows["q1_start"] = 1
course_windows["q2_start"] = course_windows["q1_end"] + 1
course_windows["q3_start"] = course_windows["q2_end"] + 1
course_windows["q4_start"] = course_windows["q3_end"] + 1

print("\nCourse windows:")
print(course_windows[COURSE_KEYS + ["course_weeks","q1_end","q2_end","q3_end","q4_end"]])



In [ ]:
# =============================================================================
# SECTION 5 — Encode studentInfo
# =============================================================================

studentInfo = studentInfo.copy()
studentInfo["target_at_risk"] = (
    studentInfo["final_result"].isin(["Fail","Withdrawn"])).astype(int)
studentInfo["gender_bin"]     = (
    studentInfo["gender"].astype(str).str.upper() == "M").astype(int)
studentInfo["disability_bin"] = (
    studentInfo["disability"].astype(str).str.upper() == "Y").astype(int)

age_map = {"0-35": 0, "35-55": 1, "55<=": 2, "55+": 2}
studentInfo["age_band_ord"] = studentInfo["age_band"].map(age_map)
studentInfo["age_band_missing_flag"] = (
    studentInfo["age_band"].isna() |
    ~studentInfo["age_band"].isin(age_map.keys())).astype(int)
studentInfo["age_band_ord"] = studentInfo["age_band_ord"].fillna(
    studentInfo["age_band_ord"].median())

edu_map = {
    "No Formal quals": 0, "Lower Than A Level": 1,
    "A Level or Equivalent": 2, "HE Qualification": 3,
    "Post Graduate Qualification": 4}
studentInfo["highest_education_ord"] = studentInfo["highest_education"].map(edu_map)
studentInfo["highest_education_missing_flag"] = (
    studentInfo["highest_education"].isna() |
    ~studentInfo["highest_education"].isin(edu_map.keys())).astype(int)
studentInfo["highest_education_ord"] = studentInfo["highest_education_ord"].fillna(
    studentInfo["highest_education_ord"].median())

def map_region_to_nation(region):
    if pd.isna(region): return "Other"
    r = str(region).lower()
    if "scotland" in r:                 return "Scotland"
    if "wales" in r:                    return "Wales"
    if "ireland" in r:                  return "Northern_Ireland"
    if "england" in r or "region" in r: return "England"
    return "Other"

studentInfo["nation"] = studentInfo["region"].apply(map_region_to_nation)
nation_ohe  = pd.get_dummies(studentInfo["nation"], prefix="nation", dummy_na=False)
studentInfo = pd.concat([studentInfo, nation_ohe], axis=1)

def bin_imd_5(x):
    x = str(x).strip()
    if x in {"0-10%","10-20%"}:   return "VeryLow"
    if x in {"20-30%","30-40%"}:  return "Low"
    if x in {"40-50%","50-60%"}:  return "Medium"
    if x in {"60-70%","70-80%"}:  return "High"
    if x in {"80-90%","90-100%"}: return "VeryHigh"
    return "Unknown"

studentInfo["imd_band"]     = studentInfo["imd_band"].fillna("Unknown").astype(str)
studentInfo["imd_5cat"]     = studentInfo["imd_band"].apply(bin_imd_5)
imd_ord_map = {"VeryLow":0,"Low":1,"Medium":2,"High":3,"VeryHigh":4,"Unknown":np.nan}
studentInfo["imd_ordinal"]  = studentInfo["imd_5cat"].map(imd_ord_map)
studentInfo["imd_unknown_flag"] = studentInfo["imd_ordinal"].isna().astype(int)
studentInfo["imd_ordinal"]  = studentInfo["imd_ordinal"].fillna(
    studentInfo["imd_ordinal"].median()).astype(float)

studentInfo.drop(
    columns=["gender","disability","age_band","highest_education",
             "region","nation","imd_band","imd_5cat"], inplace=True)


In [ ]:
# =============================================================================
# SECTION 6 — Build base_static
# =============================================================================

courses_no_weeks = courses.drop(columns=["course_weeks"], errors="ignore")
base_static = (
    studentInfo
    .merge(studentReg, on=KEYS, how="inner")
    .merge(courses_no_weeks, on=COURSE_KEYS, how="inner")
    .merge(course_windows[COURSE_KEYS + ["course_weeks","q1_end","q2_end","q3_end","q4_end"]],
           on=COURSE_KEYS, how="inner")
)
print(f"\nbase_static: {base_static.shape}  |  at-risk: {base_static['target_at_risk'].mean()*100:.1f}%")



In [ ]:
# =============================================================================
# SECTION 7 — Cross-presentation split
# Latest presentation per module (chronological) = test
# All earlier presentations = train
# =============================================================================

def parse_presentation_order(pres):
    """Parse '2013B', '2013J', '2014B' → (year, season) for chronological ordering."""
    pres = str(pres).strip()
    try:
        year         = int(pres[:4])
        season_order = {"B": 0, "J": 1}.get(pres[4:].upper(), 0)
        return (year, season_order)
    except:
        return (0, 0)

pres_per_module = (
    base_static.groupby(COURSE_KEYS).size().reset_index(name="n_students"))
pres_per_module["pres_order"] = pres_per_module["code_presentation"].apply(
    parse_presentation_order)
pres_per_module = pres_per_module.sort_values(
    ["code_module","pres_order"]).reset_index(drop=True)

print("\nAll presentations (chronological):")
print(pres_per_module[["code_module","code_presentation","n_students"]].to_string(index=False))

single_pres = (pres_per_module.groupby("code_module")["code_presentation"]
               .nunique().pipe(lambda s: s[s == 1]).index.tolist())
if single_pres:
    print(f"\n⚠️  Modules with 1 presentation (train only): {single_pres}")

# Latest presentation per module = test
test_presentations = (
    pres_per_module.sort_values(["code_module","pres_order"])
    .groupby("code_module").last().reset_index()
    [["code_module","code_presentation"]]
)
print("\nHeld-out test presentations:")
print(test_presentations.to_string(index=False))

test_set  = set(zip(test_presentations["code_module"], test_presentations["code_presentation"]))
test_mask = base_static.apply(
    lambda r: (r["code_module"], r["code_presentation"]) in test_set, axis=1)

train_keys = base_static.loc[~test_mask, KEYS].copy()
test_keys  = base_static.loc[test_mask,  KEYS].copy()

print(f"\nTrain: {len(train_keys):,} students | "
      f"{base_static.loc[~test_mask,'code_presentation'].nunique()} presentations")
print(f"Test:  {len(test_keys):,} students | "
      f"{base_static.loc[test_mask,'code_presentation'].nunique()} presentations")

assert len(
    set(zip(train_keys["code_module"], train_keys["code_presentation"])) &
    set(zip(test_keys["code_module"],  test_keys["code_presentation"]))
) == 0, "Presentation leakage detected!"
print("✓ No presentation overlap")



In [ ]:
# =============================================================================
# SECTION 8 — Process VLE to weekly level (once, shared by both cohorts)
# =============================================================================

studentVle["date"]      = pd.to_numeric(studentVle["date"],      errors="coerce")
studentVle["sum_click"] = pd.to_numeric(
    studentVle["sum_click"], errors="coerce").fillna(0).clip(lower=0)

studentVle_daily = (
    studentVle
    .groupby(["id_student","code_module","code_presentation","id_site","date"], as_index=False)
    .agg(sum_click=("sum_click","sum"))
)
vle_lookup           = vle[["id_site","activity_type"]].drop_duplicates()
studentVle_daily_enr = studentVle_daily.merge(vle_lookup, on="id_site", how="left")
del studentVle_daily, vle_lookup, studentVle
gc.collect()

studentVle_daily_enr = studentVle_daily_enr[studentVle_daily_enr["date"] >= 0].copy()
studentVle_daily_enr["week"] = (studentVle_daily_enr["date"] // 7) + 1

def map_engagement_dim(activity_type):
    if pd.isna(activity_type): return "unknown"
    a = str(activity_type).strip().lower()
    if a in CONTENT_TYPES:    return "content"
    if a in ASSESSMENT_TYPES: return "assessment"
    if a in SOCIAL_TYPES:     return "social"
    return "other"

studentVle_daily_enr["engagement_dim"] = studentVle_daily_enr["activity_type"].apply(
    map_engagement_dim)
studentVle_daily_enr = studentVle_daily_enr[
    ["id_student","code_module","code_presentation","week","sum_click","engagement_dim"]].copy()

studentVle_weekly = (
    studentVle_daily_enr
    .groupby(["id_student","code_module","code_presentation","week","engagement_dim"],
             as_index=False)
    .agg(sum_click=("sum_click","sum"))
)
print(f"studentVle_weekly shape: {studentVle_weekly.shape}")
del studentVle_daily_enr
gc.collect()

# # Outlier capping (applied once — threshold is not a normaliser)
# CAP_Q      = 0.99
# group_cols = ["code_module","code_presentation","week","engagement_dim"]
# cap_tbl = (
#     studentVle_weekly
#     .groupby(group_cols, as_index=False)["sum_click"]
#     .quantile(CAP_Q).rename(columns={"sum_click":"cap_hi"})
# )
# studentVle_weekly = studentVle_weekly.merge(cap_tbl, on=group_cols, how="left")
# studentVle_weekly["sum_click"] = np.minimum(
#     studentVle_weekly["sum_click"],
#     studentVle_weekly["cap_hi"].fillna(studentVle_weekly["sum_click"]))
# studentVle_weekly.drop(columns=["cap_hi"], inplace=True)
# del cap_tbl
# gc.collect()

# Clean assessments (once, shared)
assess_clean = assessments.copy()
assess_clean["assessment_type"] = assess_clean["assessment_type"].astype(str).str.lower()
assess_clean = assess_clean[assess_clean["assessment_type"] != "exam"].copy()
assess_clean["date"] = pd.to_numeric(assess_clean["date"], errors="coerce")
assess_clean = assess_clean.dropna(subset=["date"])
assess_clean = assess_clean[
    ["id_assessment","code_module","code_presentation","date"]
].rename(columns={"date":"due_date"})

sa = studentAssessment.copy()
sa = sa[sa["date_submitted"].notna()].copy()
sa["date_submitted"] = pd.to_numeric(sa["date_submitted"], errors="coerce").clip(lower=0)
sa = sa[["id_assessment","id_student","date_submitted"]]

sub_all = sa.merge(assess_clean, on="id_assessment", how="inner", validate="m:1")
sub_all = sub_all.merge(
    course_windows[COURSE_KEYS + ["q1_end","q2_end","q3_end","q4_end"]],
    on=COURSE_KEYS, how="inner", validate="m:1")

print("Raw data processing complete ✓")

In [ ]:
# =============================================================================
# SECTION 9 — Feature engineering functions
# Every function takes cohort_keys — all reference stats within cohort only
# =============================================================================
def cap_outliers_within_cohort(vle_weekly, cohort_keys, cap_q=0.99):
    # keep only this cohort first
    vle_c = vle_weekly.merge(cohort_keys, on=KEYS, how="inner")

    group_cols = ["code_module","code_presentation","week","engagement_dim"]

    cap_tbl = (
        vle_c.groupby(group_cols, as_index=False)["sum_click"]
        .quantile(cap_q)
        .rename(columns={"sum_click":"cap_hi"})
    )

    vle_c = vle_c.merge(cap_tbl, on=group_cols, how="left")
    vle_c["sum_click"] = np.minimum(vle_c["sum_click"], vle_c["cap_hi"].fillna(vle_c["sum_click"]))
    vle_c = vle_c.drop(columns=["cap_hi"])

    return vle_c
    
def build_weekly_norm_features(studentVle_weekly, cohort_keys, course_windows):
    """Weekly norm columns. Denominators from cohort only."""
    vle_w = studentVle_weekly[studentVle_weekly["engagement_dim"].isin(dim_keep)].copy()
    vle_w = vle_w.merge(course_windows[COURSE_KEYS+["course_weeks"]],
                        on=COURSE_KEYS, how="inner", validate="m:1")
    vle_w = vle_w[(vle_w["week"] >= 1) & (vle_w["week"] <= vle_w["course_weeks"])].copy()
    vle_w["week"] = pd.to_numeric(vle_w["week"], errors="coerce")
    vle_w = vle_w.dropna(subset=["week"]).copy()
    vle_w["week"] = vle_w["week"].astype(int)

    vle_cohort = vle_w.merge(cohort_keys, on=KEYS, how="inner")
    course_week_dim = (
        vle_cohort
        .groupby(COURSE_KEYS+["week","engagement_dim"], as_index=False)
        .agg(course_clicks=("sum_click","sum"))
    )
    vle_cohort = vle_cohort.merge(
        course_week_dim, on=COURSE_KEYS+["week","engagement_dim"],
        how="left", validate="m:1")
    vle_cohort["share_clicks"] = np.where(
        vle_cohort["course_clicks"] > 0,
        vle_cohort["sum_click"] / (vle_cohort["course_clicks"] + eps), 0.0)

    weekly_wide = (
        vle_cohort.pivot_table(
            index=KEYS, columns=["week","engagement_dim"],
            values="share_clicks", fill_value=0.0)
        .reset_index()
    )
    weekly_wide.columns = [
        c[0] if (isinstance(c,tuple) and (c[1]=="" or c[1] is None)) else c
        for c in weekly_wide.columns
    ]
    def _col_name(col):
        if isinstance(col, tuple):
            wk, dim = col
            return f"wk{int(wk):02d}_{dim}_norm"
        return col
    return weekly_wide.rename(columns=_col_name)


In [ ]:
def add_weekly_aggregates(df, course_weeks_col="course_weeks", suffix="_norm", eps=1e-9):
    """Course-length-aware aggregated features from weekly norm columns."""
    pat   = re.compile(rf"^wk(\d{{2}})_(content|assessment|social){re.escape(suffix)}$")
    weeks = sorted({int(pat.match(c).group(1)) for c in df.columns if pat.match(c)})
    if not weeks:
        return df.copy()
    max_w = max(weeks)

    Xc = df.reindex(columns=[f"wk{w:02d}_content{suffix}"    for w in weeks], fill_value=0.0).to_numpy(float)
    Xa = df.reindex(columns=[f"wk{w:02d}_assessment{suffix}"  for w in weeks], fill_value=0.0).to_numpy(float)
    Xs = df.reindex(columns=[f"wk{w:02d}_social{suffix}"      for w in weeks], fill_value=0.0).to_numpy(float)
    Xt = Xc + Xa + Xs

    cw         = pd.to_numeric(df[course_weeks_col], errors="coerce").fillna(max_w).clip(1,max_w).to_numpy(int)
    week_index = np.array(weeks, dtype=int)[None,:]
    valid_mask = (week_index <= cw[:,None]).astype(float)
    Xc_v, Xa_v, Xs_v, Xt_v = Xc*valid_mask, Xa*valid_mask, Xs*valid_mask, Xt*valid_mask

    content_eng      = Xc_v.sum(axis=1)
    assess_eng       = Xa_v.sum(axis=1)
    social_eng       = Xs_v.sum(axis=1)
    total_eng        = Xt_v.sum(axis=1)
    content_ratio    = content_eng    / (total_eng + eps)
    assessment_ratio = assess_eng     / (total_eng + eps)
    social_ratio     = social_eng     / (total_eng + eps)
    active_weeks     = (Xt_v > 0).sum(axis=1).astype(int)
    weeks_observed   = cw
    active_weeks_ratio = active_weeks / (weeks_observed + eps)

    row_sum  = Xt_v.sum(axis=1, keepdims=True)
    P        = np.divide(Xt_v, row_sum + eps)
    logP     = np.zeros_like(P)
    m        = P > 0
    logP[m]  = np.log(P[m])
    ent      = -(P * logP).sum(axis=1)
    ent_norm = ent / (np.log(weeks_observed) + eps)

    new_feat = pd.DataFrame({
        f"content_engagement{suffix}":           content_eng,
        f"assessment_engagement{suffix}":        assess_eng,
        f"social_engagement{suffix}":            social_eng,
        f"total_engagement{suffix}":             total_eng,
        f"content_ratio{suffix}":                content_ratio,
        f"assessment_ratio{suffix}":             assessment_ratio,
        f"social_ratio{suffix}":                 social_ratio,
        f"active_weeks{suffix}":                 active_weeks,
        f"weeks_observed{suffix}":               weeks_observed,
        f"active_weeks_ratio{suffix}":           active_weeks_ratio,
        f"engagement_entropy{suffix}":           ent,
        f"engagement_week_entropy_norm{suffix}": ent_norm,
    }, index=df.index)
    return pd.concat([df.copy(), new_feat], axis=1).copy()


In [ ]:
def build_quarter_norm_features(studentVle_weekly, cohort_keys,
                                 course_windows, q_end_col, q_label):
    """Quarter engagement norm features. Denominators from cohort only."""
    vle_q = studentVle_weekly.merge(
        course_windows[COURSE_KEYS+["course_weeks","q1_end","q2_end","q3_end","q4_end"]],
        on=COURSE_KEYS, how="inner", validate="m:1").copy()
    vle_q = vle_q[(vle_q["week"] >= 1) & (vle_q["week"] <= vle_q["course_weeks"])].copy()

    df        = vle_q[vle_q["week"] <= vle_q[q_end_col]].copy()
    df_cohort = df.merge(cohort_keys, on=KEYS, how="inner")

    course_dim = (
        df_cohort[df_cohort["engagement_dim"].isin(dim_keep)]
        .groupby(COURSE_KEYS+["engagement_dim"], as_index=False)
        .agg(course_clicks=("sum_click","sum"))
    )
    course_total = (
        df_cohort.groupby(COURSE_KEYS, as_index=False)
        .agg(course_total_clicks=("sum_click","sum"))
    )
    student_dim = (
        df[df["engagement_dim"].isin(dim_keep)]
        .merge(cohort_keys, on=KEYS, how="inner")
        .groupby(KEYS+["engagement_dim"], as_index=False)
        .agg(clicks=("sum_click","sum"))
    )
    student_total = (
        df.merge(cohort_keys, on=KEYS, how="inner")
        .groupby(KEYS, as_index=False)
        .agg(total_clicks=("sum_click","sum"))
    )

    student_dim = student_dim.merge(
        course_dim, on=COURSE_KEYS+["engagement_dim"], how="left", validate="m:1")
    student_dim["norm"] = np.where(
        student_dim["course_clicks"] > 0,
        student_dim["clicks"] / student_dim["course_clicks"], 0.0)

    dim_wide = (
        student_dim
        .pivot_table(index=KEYS, columns="engagement_dim", values="norm", fill_value=0.0)
        .reset_index()
        .rename(columns={
            "content":    f"norm_content_{q_label}",
            "assessment": f"norm_assessment_{q_label}",
            "social":     f"norm_social_{q_label}",
        })
    )
    total_norm = student_total.merge(course_total, on=COURSE_KEYS, how="left", validate="m:1")
    total_norm[f"norm_total_{q_label}"] = np.where(
        total_norm["course_total_clicks"] > 0,
        total_norm["total_clicks"] / total_norm["course_total_clicks"], 0.0)
    total_norm = total_norm[KEYS + [f"norm_total_{q_label}"]]

    out = total_norm.merge(dim_wide, on=KEYS, how="left")
    for c in [f"norm_content_{q_label}", f"norm_assessment_{q_label}", f"norm_social_{q_label}"]:
        out[c] = out.get(c, pd.Series(0.0, index=out.index)).fillna(0.0)
    return out



In [ ]:
# def add_weekly_aggregates(df, course_weeks_col="course_weeks", suffix="_norm", eps=1e-9):
#     """Course-length-aware aggregated features from weekly norm columns."""
#     pat   = re.compile(rf"^wk(\d{{2}})_(content|assessment|social){re.escape(suffix)}$")
#     weeks = sorted({int(pat.match(c).group(1)) for c in df.columns if pat.match(c)})
#     if not weeks:
#         return df.copy()
#     max_w = max(weeks)

#     Xc = df.reindex(columns=[f"wk{w:02d}_content{suffix}"    for w in weeks], fill_value=0.0).to_numpy(float)
#     Xa = df.reindex(columns=[f"wk{w:02d}_assessment{suffix}"  for w in weeks], fill_value=0.0).to_numpy(float)
#     Xs = df.reindex(columns=[f"wk{w:02d}_social{suffix}"      for w in weeks], fill_value=0.0).to_numpy(float)
#     Xt = Xc + Xa + Xs

#     cw         = pd.to_numeric(df[course_weeks_col], errors="coerce").fillna(max_w).clip(1,max_w).to_numpy(int)
#     week_index = np.array(weeks, dtype=int)[None,:]
#     valid_mask = (week_index <= cw[:,None]).astype(float)
#     Xc_v, Xa_v, Xs_v, Xt_v = Xc*valid_mask, Xa*valid_mask, Xs*valid_mask, Xt*valid_mask

#     content_eng      = Xc_v.sum(axis=1)
#     assess_eng       = Xa_v.sum(axis=1)
#     social_eng       = Xs_v.sum(axis=1)
#     total_eng        = Xt_v.sum(axis=1)
#     content_ratio    = content_eng    / (total_eng + eps)
#     assessment_ratio = assess_eng     / (total_eng + eps)
#     social_ratio     = social_eng     / (total_eng + eps)
#     active_weeks     = (Xt_v > 0).sum(axis=1).astype(int)
#     weeks_observed   = cw
#     active_weeks_ratio = active_weeks / (weeks_observed + eps)

#     row_sum  = Xt_v.sum(axis=1, keepdims=True)
#     P        = np.divide(Xt_v, row_sum + eps)
#     logP     = np.zeros_like(P)
#     m        = P > 0
#     logP[m]  = np.log(P[m])
#     ent      = -(P * logP).sum(axis=1)
#     ent_norm = ent / (np.log(weeks_observed) + eps)

#     new_feat = pd.DataFrame({
#         f"content_engagement{suffix}":           content_eng,
#         f"assessment_engagement{suffix}":        assess_eng,
#         f"social_engagement{suffix}":            social_eng,
#         f"total_engagement{suffix}":             total_eng,
#         f"content_ratio{suffix}":                content_ratio,
#         f"assessment_ratio{suffix}":             assessment_ratio,
#         f"social_ratio{suffix}":                 social_ratio,
#         f"active_weeks{suffix}":                 active_weeks,
#         f"weeks_observed{suffix}":               weeks_observed,
#         f"active_weeks_ratio{suffix}":           active_weeks_ratio,
#         f"engagement_entropy{suffix}":           ent,
#         f"engagement_week_entropy_norm{suffix}": ent_norm,
#     }, index=df.index)
#     return pd.concat([df.copy(), new_feat], axis=1).copy()


In [ ]:
def build_ap_full(sub_all, cohort_keys, eps=1e-9):
    """Full-course AP score. Smin from cohort only."""
    sub_c  = sub_all.merge(cohort_keys, on=KEYS, how="inner")
    smin   = sub_c.groupby(COURSE_KEYS+["id_assessment"], as_index=False).agg(Smin=("date_submitted","min"))
    sub_c  = sub_c.merge(smin, on=COURSE_KEYS+["id_assessment"], how="left", validate="m:1")
    sub_c["Wa"]        = sub_c["due_date"] - sub_c["Smin"]
    sub_c["delta"]     = sub_c["due_date"] - sub_c["date_submitted"]
    sub_c["Ts"]        = np.where(sub_c["Wa"] > 0, sub_c["delta"] / (sub_c["Wa"] + eps), 0.0)
    sub_c["late_flag"] = (sub_c["date_submitted"] > sub_c["due_date"]).astype(int)
    return (
        sub_c.groupby(KEYS, as_index=False)
        .agg(AP_full=("Ts","mean"), n_submissions=("Ts","count"),
             num_late_submissions=("late_flag","sum"))
    )

In [ ]:
def build_ap_quarter(sub_all, cohort_keys, q_end_col, q_label,
                     prev_q_end_col=None, eps=1e-9):
    """Quarter AP score. Smin_q from cohort only."""
    sub_c        = sub_all.merge(cohort_keys, on=KEYS, how="inner")
    q_end_day    = sub_c[q_end_col] * 7 - 1
    prev_end_day = -1 if prev_q_end_col is None else sub_c[prev_q_end_col] * 7 - 1

    dfq = sub_c[
        (sub_c["due_date"] > prev_end_day) & (sub_c["due_date"] <= q_end_day)
    ].copy()
    dfq["late_q"] = (dfq["date_submitted"] > dfq["due_date"]).astype(int)

    smin_q = (
        dfq[dfq["date_submitted"].notna()]
        .groupby(COURSE_KEYS+["id_assessment"], as_index=False)
        .agg(Smin_q=("date_submitted","min"))
    )
    dfq         = dfq.merge(smin_q, on=COURSE_KEYS+["id_assessment"], how="left", validate="m:1")
    dfq["Wa_q"]    = dfq["due_date"] - dfq["Smin_q"]
    dfq["delta_q"] = dfq["due_date"] - dfq["date_submitted"]
    dfq["Ts_q"]    = np.where(
        (dfq["Wa_q"] > 0) & (dfq["date_submitted"].notna()),
        dfq["delta_q"] / (dfq["Wa_q"] + eps), 0.0)
    return (
        dfq.groupby(KEYS, as_index=False).agg(**{
            f"AP_{q_label}":                  ("Ts_q",   "mean"),
            f"n_submissions_{q_label}":        ("Ts_q",   "count"),
            f"num_late_submissions_{q_label}": ("late_q", "sum"),
        })
    )


In [ ]:
# def add_quarter_derived_features(df, q_label, eps=1e-9):
#     """Ratios, entropy, AP flags from quarter norm columns."""
#     t, c = f"norm_total_{q_label}", f"norm_content_{q_label}"
#     a, s = f"norm_assessment_{q_label}", f"norm_social_{q_label}"
#     for col in [t, c, a, s]:
#         if col not in df.columns: df[col] = 0.0
#         df[col] = df[col].fillna(0.0)

#     df[f"content_ratio_{q_label}"]    = df[c] / (df[t] + eps)
#     df[f"assessment_ratio_{q_label}"] = df[a] / (df[t] + eps)
#     df[f"social_ratio_{q_label}"]     = df[s] / (df[t] + eps)

#     vals    = df[[c, a, s]].to_numpy(float)
#     row_sum = vals.sum(axis=1, keepdims=True)
#     P       = vals / (row_sum + eps)
#     logP    = np.zeros_like(P)
#     mask    = P > 0
#     logP[mask] = np.log(P[mask])
#     dim_ent = -(P * logP).sum(axis=1)
#     df[f"dim_entropy_{q_label}"]      = dim_ent
#     df[f"dim_entropy_norm_{q_label}"] = dim_ent / (np.log(3) + eps)

#     ap = f"AP_{q_label}"
#     if ap in df.columns:
#         df[f"{ap}_missing"] = df[ap].isna().astype(int)
#         df[ap]              = df[ap].fillna(0.0)
#     return df
def add_quarter_derived_features(df, q_label, eps=1e-9):
    t, c = f"norm_total_{q_label}", f"norm_content_{q_label}"
    a, s = f"norm_assessment_{q_label}", f"norm_social_{q_label}"
    for col in [t, c, a, s]:
        if col not in df.columns: df[col] = 0.0
        df[col] = df[col].fillna(0.0)

    # FIX: use sum of the three kept dimensions as denominator, not norm_total
    # norm_total includes 'other'/'unknown' which are excluded from c/a/s
    dim_sum = df[c] + df[a] + df[s]

    df[f"content_ratio_{q_label}"]    = df[c] / (dim_sum + eps)
    df[f"assessment_ratio_{q_label}"] = df[a] / (dim_sum + eps)
    df[f"social_ratio_{q_label}"]     = df[s] / (dim_sum + eps)

    # entropy over the three dimensions — unchanged
    vals    = df[[c, a, s]].to_numpy(float)
    row_sum = vals.sum(axis=1, keepdims=True)
    P       = vals / (row_sum + eps)
    logP    = np.zeros_like(P)
    mask    = P > 0
    logP[mask] = np.log(P[mask])
    dim_ent = -(P * logP).sum(axis=1)
    df[f"dim_entropy_{q_label}"]      = dim_ent
    df[f"dim_entropy_norm_{q_label}"] = dim_ent / (np.log(3) + eps)

    ap = f"AP_{q_label}"
    if ap in df.columns:
        df[f"{ap}_missing"] = df[ap].isna().astype(int)
        df[ap]              = df[ap].fillna(0.0)
    return df

In [ ]:
def build_spacing_scores(studentVle_weekly_cohort, course_windows, eps=1e-9):
    """
    Spacing scores per quarter + full course.
    Uses only WHEN a student engages — no cross-student normalisation needed.
    Pass VLE data pre-filtered to the cohort.
    """
    cw = course_windows.copy()
    cw["full_start"] = 1
    cw["full_end"]   = cw["q4_end"]

    q_windows = [
        ("q1_start","q1_end","Q1"), ("q2_start","q2_end","Q2"),
        ("q3_start","q3_end","Q3"), ("q4_start","q4_end","Q4"),
        ("full_start","full_end","full"),
    ]
    results = {}
    for q_start_col, q_end_col, q_label in q_windows:
        vle_q = studentVle_weekly_cohort.merge(
            cw[COURSE_KEYS+["course_weeks",q_start_col,q_end_col]],
            on=COURSE_KEYS, how="inner", validate="m:1").copy()
        vle_q = vle_q[
            (vle_q["week"] >= vle_q[q_start_col]) &
            (vle_q["week"] <= vle_q[q_end_col]) &
            (vle_q["engagement_dim"].isin(dim_keep))
        ].copy()

        student_week = (
            vle_q.groupby(KEYS+["week"], as_index=False)
            .agg(total_click=("sum_click","sum"))
        )
        bounds  = cw[COURSE_KEYS+[q_start_col,q_end_col]].drop_duplicates()
        records = []

        for key, grp in student_week.groupby(KEYS):
            id_student, code_module, code_presentation = key
            b = bounds[
                (bounds["code_module"] == code_module) &
                (bounds["code_presentation"] == code_presentation)]
            if b.empty:
                records.append({
                    "id_student": id_student, "code_module": code_module,
                    "code_presentation": code_presentation,
                    f"spacing_score_{q_label}": 0.0, f"spacing_defined_{q_label}": 0})
                continue

            s, e  = int(b[q_start_col].iloc[0]), int(b[q_end_col].iloc[0])
            qlen  = e - s + 1
            weeks_active = grp.loc[grp["total_click"] > 0, "week"].sort_values().to_numpy()
            m = len(weeks_active)

            if m < 2 or qlen < 2:
                score, defined = 0.0, 0
            else:
                gaps           = np.diff(weeks_active)
                ideal_gap      = (qlen - 1) / (m - 1)
                dev            = np.mean(np.abs(gaps - ideal_gap)) / (ideal_gap + eps)
                consistency    = 1.0 / (1.0 + dev)
                break_severity = (gaps.max() - 1) / max(qlen - 1, 1)
                score          = 0.7 * consistency + 0.3 * (1.0 - np.clip(break_severity, 0.0, 1.0))
                defined        = 1

            records.append({
                "id_student": id_student, "code_module": code_module,
                "code_presentation": code_presentation,
                f"spacing_score_{q_label}": score, f"spacing_defined_{q_label}": defined})

        results[q_label] = pd.DataFrame(records)
    return results


In [ ]:
# def build_quarter_temporal_features(master_df, course_windows):
#     """active_weeks_ratio_Qx and week_entropy_norm_Qx from weekly norm columns."""
#     pat        = re.compile(r"^wk(\d{2})_(content|assessment|social)_norm$")
#     weeks      = sorted({int(pat.match(c).group(1))
#                          for c in master_df.columns if pat.match(c)})
#     week_index = np.array(weeks)

#     Xt = (
#         master_df[[f"wk{w:02d}_content_norm"    for w in weeks]].to_numpy(float) +
#         master_df[[f"wk{w:02d}_assessment_norm"  for w in weeks]].to_numpy(float) +
#         master_df[[f"wk{w:02d}_social_norm"      for w in weeks]].to_numpy(float)
#     )

#     q_end_cols = ["q1_end","q2_end","q3_end","q4_end"]
#     missing_q  = [c for c in q_end_cols if c not in master_df.columns]
#     if missing_q:
#         master_df = master_df.merge(
#             course_windows[COURSE_KEYS + q_end_cols].drop_duplicates(),
#             on=COURSE_KEYS, how="left")

#     extra = master_df[KEYS].copy()
#     for q_label, q_end_col in [("Q1","q1_end"),("Q2","q2_end"),("Q3","q3_end"),("Q4","q4_end")]:
#         q_end   = master_df[q_end_col].to_numpy(int)
#         valid   = (week_index[None,:] <= q_end[:,None])
#         Xt_q    = np.where(valid, Xt, 0.0)

#         active_weeks = (Xt_q > 0).sum(axis=1)
#         n_valid      = valid.sum(axis=1).clip(min=1)
#         extra[f"active_weeks_ratio_{q_label}"] = active_weeks / n_valid

#         row_sum = Xt_q.sum(axis=1, keepdims=True)
#         P       = Xt_q / (row_sum + eps)
#         logP    = np.zeros_like(P)
#         m       = P > 0
#         logP[m] = np.log(P[m])
#         ent     = -(P * logP).sum(axis=1)
#         extra[f"week_entropy_norm_{q_label}"] = ent / (np.log(n_valid) + eps)

#     return extra, master_df
def build_quarter_temporal_features(master_df, course_windows, eps=1e-9):
    """
    Build:
      - active_weeks_ratio_Qx
      - week_entropy_norm_Qx

    Uses weekly norm columns safely via reindex(fill_value=0.0),
    so missing week-dimension columns do not break the pipeline.
    """
    pat = re.compile(r"^wk(\d{2})_(content|assessment|social)_norm$")
    weeks = sorted({
        int(pat.match(c).group(1))
        for c in master_df.columns
        if pat.match(c)
    })

    if not weeks:
        extra = master_df[KEYS].copy()
        for q_label in ["Q1", "Q2", "Q3", "Q4"]:
            extra[f"active_weeks_ratio_{q_label}"] = 0.0
            extra[f"week_entropy_norm_{q_label}"] = 0.0
        return extra, master_df

    # Ensure quarter cutoff columns exist
    q_end_cols = ["q1_end", "q2_end", "q3_end", "q4_end"]
    missing_q = [c for c in q_end_cols if c not in master_df.columns]
    if missing_q:
        master_df = master_df.merge(
            course_windows[COURSE_KEYS + q_end_cols].drop_duplicates(),
            on=COURSE_KEYS,
            how="left"
        )

    # Safe reindex in case some weekly columns are missing
    Xc = master_df.reindex(
        columns=[f"wk{w:02d}_content_norm" for w in weeks],
        fill_value=0.0
    ).to_numpy(float)

    Xa = master_df.reindex(
        columns=[f"wk{w:02d}_assessment_norm" for w in weeks],
        fill_value=0.0
    ).to_numpy(float)

    Xs = master_df.reindex(
        columns=[f"wk{w:02d}_social_norm" for w in weeks],
        fill_value=0.0
    ).to_numpy(float)

    Xt = Xc + Xa + Xs
    week_index = np.array(weeks)

    extra = master_df[KEYS].copy()

    for q_label, q_end_col in [
        ("Q1", "q1_end"),
        ("Q2", "q2_end"),
        ("Q3", "q3_end"),
        ("Q4", "q4_end"),
    ]:
        q_end = pd.to_numeric(master_df[q_end_col], errors="coerce").fillna(0).astype(int).to_numpy()
        valid = (week_index[None, :] <= q_end[:, None])

        Xt_q = np.where(valid, Xt, 0.0)

        active_weeks = (Xt_q > 0).sum(axis=1)
        n_valid = valid.sum(axis=1).clip(min=1)

        extra[f"active_weeks_ratio_{q_label}"] = active_weeks / (n_valid + eps)

        row_sum = Xt_q.sum(axis=1, keepdims=True)
        P = Xt_q / (row_sum + eps)

        logP = np.zeros_like(P)
        m = P > 0
        logP[m] = np.log(P[m])

        ent = -(P * logP).sum(axis=1)
        extra[f"week_entropy_norm_{q_label}"] = ent / (np.log(n_valid) + eps)

    return extra, master_df

In [ ]:
def safe_cols(df, cols):
    return [c for c in cols if c in df.columns]


In [ ]:
# =============================================================================
# SECTION 10 — Feature sets
# =============================================================================

DEMOGRAPHIC_FEATURES = [
    "gender_bin", "disability_bin",
    "age_band_ord", "age_band_missing_flag",
    "highest_education_ord", "highest_education_missing_flag",
    "imd_ordinal", "imd_unknown_flag",
    "nation_England", "nation_Scotland", "nation_Wales", "nation_Northern_Ireland",
    "late_registration_flag",
]
ACADEMIC_FEATURES = [
    "studied_credits", "course_type_stem", "num_of_prev_attempts",
]
MASTER_BEHAVIOURAL = [
    "total_engagement_norm", "content_engagement_norm",
    "assessment_engagement_norm", "social_engagement_norm",
    "content_ratio_norm", "assessment_ratio_norm", "social_ratio_norm",
    "active_weeks_ratio_norm", "engagement_week_entropy_norm_norm",
    "engagement_entropy_norm",
    "AP_full", "AP_full_missing", "n_submissions", "num_late_submissions",
    "spacing_score_full", "spacing_defined_full"
]

def quarter_behavioural(q):
    return [
        f"norm_total_{q}", f"norm_content_{q}",
        f"norm_assessment_{q}", f"norm_social_{q}",
        f"content_ratio_{q}", f"assessment_ratio_{q}", f"social_ratio_{q}",
        f"dim_entropy_norm_{q}",
        f"active_weeks_ratio_{q}", f"week_entropy_norm_{q}",
        f"AP_{q}", f"AP_{q}_missing",
        f"n_submissions_{q}", f"num_late_submissions_{q}",
        f"spacing_score_{q}", f"spacing_defined_{q}",
    ]


In [ ]:
# =============================================================================
# SECTION 11 — Master feature engineering function (runs per cohort)
# =============================================================================

def build_cohort_features(cohort_keys, studentVle_weekly, sub_all,
                           base_static, course_windows, label="cohort"):
    """
    Build complete ML-ready feature tables for one cohort.
    All reference stats computed from this cohort only. No outcome labels used.
    Returns: ml_master, ml_Q1, ml_Q2, ml_Q3, ml_Q4
    """
    print(f"\n  [{label}] {len(cohort_keys):,} students...")
    base = base_static.merge(cohort_keys, on=KEYS, how="inner")
    studentVle_weekly_c = cap_outliers_within_cohort(studentVle_weekly, cohort_keys, cap_q=0.99)
    assert len(base) == len(cohort_keys), "Student count mismatch after base merge"


    # Weekly norm
    weekly_wide      = build_weekly_norm_features(studentVle_weekly_c, cohort_keys, course_windows)
    master           = base.merge(weekly_wide, on=KEYS, how="left")
    norm_weekly_cols = [c for c in weekly_wide.columns if c not in KEYS]
    master[norm_weekly_cols] = master[norm_weekly_cols].fillna(0.0)
    master = add_weekly_aggregates(master, course_weeks_col="course_weeks", suffix="_norm", eps=eps)

    # AP full
    ap_full = build_ap_full(sub_all, cohort_keys, eps)
    master  = master.merge(ap_full, on=KEYS, how="left")
    master["AP_full_missing"]      = master["AP_full"].isna().astype(int)
    master["AP_full"]              = master["AP_full"].fillna(0.0)
    master["n_submissions"]        = master["n_submissions"].fillna(0).astype(int)
    master["num_late_submissions"] = master["num_late_submissions"].fillna(0).astype(int)

    # Quarter tables
    quarter_tables = {}
    for q_end_col, q_label, prev_q_end_col in [
        ("q1_end","Q1",None), ("q2_end","Q2","q1_end"),
        ("q3_end","Q3","q2_end"), ("q4_end","Q4","q3_end")
    ]:
        q_norm = build_quarter_norm_features(
            studentVle_weekly_c, cohort_keys, course_windows, q_end_col, q_label)
        ap_q   = build_ap_quarter(
            sub_all, cohort_keys, q_end_col, q_label, prev_q_end_col, eps)
        q_df   = add_quarter_derived_features(
            base.merge(q_norm, on=KEYS, how="left").merge(ap_q, on=KEYS, how="left"),
            q_label, eps)
        quarter_tables[q_label] = q_df

    # Spacing
    vle_cohort = studentVle_weekly_c.merge(cohort_keys, on=KEYS, how="inner")
    spacing    = build_spacing_scores(vle_cohort, course_windows, eps)

    spacing_all = spacing["Q1"]
    for q in ["Q2","Q3","Q4"]:
        spacing_all = spacing_all.merge(spacing[q], on=KEYS, how="outer")
    for q in ["Q1","Q2","Q3","Q4"]:
        spacing_all[f"spacing_score_{q}"]   = spacing_all[f"spacing_score_{q}"].fillna(0.0)
        spacing_all[f"spacing_defined_{q}"] = spacing_all[f"spacing_defined_{q}"].fillna(0).astype(int)
    master = master.merge(spacing_all, on=KEYS, how="left")
    # ✅ add this (post-merge fill)
    for q in ["Q1","Q2","Q3","Q4"]:
        master[f"spacing_score_{q}"]   = master[f"spacing_score_{q}"].fillna(0.0)
        master[f"spacing_defined_{q}"] = master[f"spacing_defined_{q}"].fillna(0).astype(int)

    if "full" in spacing:
        master = master.merge(
            spacing["full"][KEYS+["spacing_score_full","spacing_defined_full"]],
            on=KEYS, how="left")
        master["spacing_score_full"]   = master["spacing_score_full"].fillna(0.0)
        master["spacing_defined_full"] = master["spacing_defined_full"].fillna(0).astype(int)

    for q_label in ["Q1","Q2","Q3","Q4"]:
        q_df = quarter_tables[q_label]
        old  = [c for c in q_df.columns if "spacing" in c and q_label in c]
        q_df.drop(columns=old, errors="ignore", inplace=True)
        q_df = q_df.merge(spacing[q_label], on=KEYS, how="left", validate="1:1")
        q_df[f"spacing_score_{q_label}"]   = q_df[f"spacing_score_{q_label}"].fillna(0.0)
        q_df[f"spacing_defined_{q_label}"] = q_df[f"spacing_defined_{q_label}"].fillna(0).astype(int)
        quarter_tables[q_label] = q_df

    # Drop leakage columns
    master = master.drop(columns=[
        "final_result","outcome_group","risk_label",
        "module_presentation_length","weeks_observed_norm",
        "date_registration","date_unregistration",
        "unregistered_flag","unregistration_week",
    ], errors="ignore")

    # Quarter temporal features
    quarter_extra, master = build_quarter_temporal_features(master, course_windows)

    # Build final ML tables
    def build_ml_master(mt):
        dem, acad, beh = (safe_cols(mt, DEMOGRAPHIC_FEATURES),
                          safe_cols(mt, ACADEMIC_FEATURES),
                          safe_cols(mt, MASTER_BEHAVIOURAL))
        ml = mt[KEYS + ["target_at_risk"] + dem + acad + beh].copy()
        ml[dem + acad + beh] = ml[dem + acad + beh].fillna(0.0)
        return ml

    def build_ml_quarter(q_df, q_label):
        new_feats    = [f"active_weeks_ratio_{q_label}", f"week_entropy_norm_{q_label}"]
        beh          = safe_cols(q_df, quarter_behavioural(q_label))
        dem          = safe_cols(q_df, DEMOGRAPHIC_FEATURES)
        acad         = safe_cols(q_df, ACADEMIC_FEATURES)
        existing_beh = [c for c in beh if c not in new_feats]
        ml           = q_df[KEYS + ["target_at_risk"] + dem + acad + existing_beh].copy()
        ml           = ml.merge(quarter_extra[KEYS + new_feats], on=KEYS, how="left")
        all_feat     = dem + acad + existing_beh + new_feats
        ml[all_feat] = ml[all_feat].fillna(0.0)
        return ml

    ml_master = build_ml_master(master)
    ml_Q1     = build_ml_quarter(quarter_tables["Q1"], "Q1")
    ml_Q2     = build_ml_quarter(quarter_tables["Q2"], "Q2")
    ml_Q3     = build_ml_quarter(quarter_tables["Q3"], "Q3")
    ml_Q4     = build_ml_quarter(quarter_tables["Q4"], "Q4")

    # ── Sequence table for LSTM — keep weekly norm columns ───────────────────
    wk_pattern = re.compile(r"^wk\d{2}_(content|assessment|social)_norm$")
    wk_cols    = [c for c in master.columns if wk_pattern.match(c)]
    ml_seq     = master[KEYS + ["target_at_risk"] + wk_cols].copy()
    ml_seq[wk_cols] = ml_seq[wk_cols].fillna(0.0)

    print(f"  master: {ml_master.shape}  at-risk={ml_master['target_at_risk'].mean()*100:.1f}%")
    print(f"  Q1:{ml_Q1.shape} Q2:{ml_Q2.shape} Q3:{ml_Q3.shape} Q4:{ml_Q4.shape}")
    print(f"  seq: {ml_seq.shape}  weekly cols: {len(wk_cols)}")
    return ml_master, ml_Q1, ml_Q2, ml_Q3, ml_Q4, ml_seq


In [24]:
# =============================================================================
# SECTION 12 — Run feature engineering for train and test cohorts independently
# =============================================================================

print("\n" + "="*70)
print("BUILDING TRAIN COHORT FEATURES")
print("="*70)
ml_master_train, ml_Q1_train, ml_Q2_train, ml_Q3_train, ml_Q4_train, ml_seq_train = \
    build_cohort_features(train_keys, studentVle_weekly, sub_all,
                           base_static, course_windows, label="Train")


print("\n" + "="*70)
print("BUILDING TEST COHORT FEATURES")
print("="*70)
ml_master_test, ml_Q1_test, ml_Q2_test, ml_Q3_test, ml_Q4_test, ml_seq_test = \
    build_cohort_features(test_keys, studentVle_weekly, sub_all,
                           base_static, course_windows, label="Test")


KeyboardInterrupt: 

In [ ]:
import re
pat      = re.compile(r"^wk(\d{2})_(content|assessment|social)_norm$")
wk_cols  = [c for c in ml_seq_train.columns if pat.match(c)]
weeks    = sorted({int(pat.match(c).group(1)) for c in wk_cols})

print(f"ml_seq_train shape : {ml_seq_train.shape}")
print(f"ml_seq_test  shape : {ml_seq_test.shape}")
print(f"Weekly cols found  : {len(wk_cols)}")
print(f"Weeks              : {weeks}")

In [ ]:
# =============================================================================
# SECTION 13 — Align columns (train and test must be identical)
# =============================================================================

def align_columns(train_df, test_df):
    train_feats = [c for c in train_df.columns if c not in KEYS + ["target_at_risk"]]
    for c in train_feats:
        if c not in test_df.columns:
            test_df[c] = 0.0
    keep = KEYS + ["target_at_risk"] + train_feats
    return train_df, test_df[[c for c in keep if c in test_df.columns]].copy()

ml_master_train, ml_master_test = align_columns(ml_master_train, ml_master_test)
ml_Q1_train,     ml_Q1_test     = align_columns(ml_Q1_train,     ml_Q1_test)
ml_Q2_train,     ml_Q2_test     = align_columns(ml_Q2_train,     ml_Q2_test)
ml_Q3_train,     ml_Q3_test     = align_columns(ml_Q3_train,     ml_Q3_test)
ml_Q4_train,     ml_Q4_test     = align_columns(ml_Q4_train,     ml_Q4_test)
print("\nColumn alignment complete ✓")


In [ ]:
# =============================================================================
# SECTION 15 — Summary
# =============================================================================

def get_feature_cols(df):
    return [c for c in df.columns if c not in KEYS + ["target_at_risk"]]

print("\n" + "="*70)
print("PIPELINE SUMMARY")
print("="*70)
print(f"\nProtocol: Cross-presentation deployment")
print(f"\nTest presentations (held out):")
print(test_presentations.to_string(index=False))
print(f"\nTrain: {len(train_keys):,} students  |  Test: {len(test_keys):,} students")
print(f"\nFeature counts (train):")
print(f"  master: {len(get_feature_cols(ml_master_train))}")
print(f"  Q1: {len(get_feature_cols(ml_Q1_train))}  "
      f"Q2: {len(get_feature_cols(ml_Q2_train))}  "
      f"Q3: {len(get_feature_cols(ml_Q3_train))}  "
      f"Q4: {len(get_feature_cols(ml_Q4_train))}")
print(f"\nAt-risk rates:")
print(f"  Train master: {ml_master_train['target_at_risk'].mean()*100:.1f}%")
print(f"  Test  master: {ml_master_test['target_at_risk'].mean()*100:.1f}%")
print("\nReady for ML pipeline ✓")


In [ ]:
# =============================================================================
# SECTION 16 — Inspect all table columns
# =============================================================================
tables = {
    "ml_master_train": ml_master_train,
    "ml_Q1_train":     ml_Q1_train,
    "ml_Q2_train":     ml_Q2_train,
    "ml_Q3_train":     ml_Q3_train,
    "ml_Q4_train":     ml_Q4_train,
}

for name, df in tables.items():
    feat_cols = [c for c in df.columns if c not in KEYS + ["target_at_risk"]]
    print(f"\n{'='*60}")
    print(f"{name}  |  shape: {df.shape}  |  {len(feat_cols)} features")
    print(f"{'='*60}")
    print(f"  KEYS + target : {KEYS + ['target_at_risk']}")
    print(f"  Features ({len(feat_cols)}):")
    for i, c in enumerate(feat_cols, 1):
        print(f"    {i:>3}. {c}")

## Clustering

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

CLUSTERING_FEATURES = [
    # Engagement profile — what type of activity (ratios only, drop norms)
    "content_ratio_norm",
    "assessment_ratio_norm",
    "social_ratio_norm",

    # Consistency — how regularly they engage
    "active_weeks_ratio_norm",
    "spacing_score_full",

    # Submission behaviour
    "AP_full",
    "num_late_submissions",
]
cluster_models = {}
CLUSTER_FEATURES_Q = {
    "Q1": [
        # "norm_content_Q1",
        # "norm_assessment_Q1", "norm_social_Q1",
        "content_ratio_Q1", "assessment_ratio_Q1", "social_ratio_Q1",
        # "dim_entropy_norm_Q1",
        "AP_Q1", 
         "num_late_submissions_Q1",
        "spacing_score_Q1", "active_weeks_ratio_Q1",
        # "week_entropy_norm_Q1",
    ],
    "Q2": [
        # "norm_content_Q2",
        # "norm_assessment_Q2", "norm_social_Q2",
        "content_ratio_Q2", "assessment_ratio_Q2", "social_ratio_Q2",
        # "dim_entropy_norm_Q2",
        "AP_Q2", 
        "num_late_submissions_Q2",
        "spacing_score_Q2", "active_weeks_ratio_Q2",
        # "week_entropy_norm_Q2",
    ],
    "Q3": [
        # "norm_content_Q3",
        # "norm_assessment_Q3", "norm_social_Q3",
        "content_ratio_Q3", "assessment_ratio_Q3", "social_ratio_Q3",
        # "dim_entropy_norm_Q3",
        "AP_Q3", 
        "num_late_submissions_Q3",
        "spacing_score_Q3", "active_weeks_ratio_Q3",
        # "week_entropy_norm_Q3",
    ],
    "Q4": [
        # "norm_content_Q4",
        # "norm_assessment_Q4", "norm_social_Q4",
        "content_ratio_Q4", "assessment_ratio_Q4", "social_ratio_Q4",
        # "dim_entropy_norm_Q4",
        "AP_Q4", 
        "num_late_submissions_Q4",
        "spacing_score_Q4", "active_weeks_ratio_Q4",
        # "week_entropy_norm_Q4",
    ],
}

# -----------------------------
# Helpers
# -----------------------------
def safe_cols(df, cols):
    """Return list of cols that exist in df (preserve order)."""
    return [c for c in cols if c in df.columns]


def _ensure_numeric_matrix(df, features):
    """
    Extract features matrix as float32, coercing non-numeric to NaN then fill with 0.
    """
    X = df[features].copy()
    for c in features:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    X = X.fillna(0.0).astype(np.float32)
    return X


def choose_k_kmeans(X_scaled, k_min=2, k_max=12, n_init=20, random_state=42):
    """
    Pick k using silhouette (primary), with CH (higher better) and DB (lower better) tracked.
    Returns:
      best_k, metrics_df
    """
    rows = []
    n = X_scaled.shape[0]

    # If too few samples, fall back safely
    if n < (k_min + 1):
        return 1, pd.DataFrame([{
            "k": 1, "silhouette": np.nan, "calinski_harabasz": np.nan, "davies_bouldin": np.nan
        }])

    k_max_eff = min(k_max, n - 1)
    k_min_eff = min(k_min, k_max_eff)

    best_k = None
    best_s = -np.inf

    for k in range(k_min_eff, k_max_eff + 1):
        km = KMeans(n_clusters=k, n_init=n_init, random_state=random_state)
        labels = km.fit_predict(X_scaled)

        # Guard: silhouette requires >=2 clusters and no empty clusters
        if len(np.unique(labels)) < 2:
            s = np.nan
        else:
            s = silhouette_score(X_scaled, labels)

        ch = calinski_harabasz_score(X_scaled, labels) if len(np.unique(labels)) > 1 else np.nan
        db = davies_bouldin_score(X_scaled, labels) if len(np.unique(labels)) > 1 else np.nan

        rows.append({"k": k, "silhouette": s, "calinski_harabasz": ch, "davies_bouldin": db})

        # Primary criterion: silhouette
        if np.isfinite(s) and s > best_s:
            best_s = s
            best_k = k

    metrics_df = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)

    # If silhouette failed everywhere (rare), choose k with best (lowest) DB, else k_min
    if best_k is None:
        if metrics_df["davies_bouldin"].notna().any():
            best_k = int(metrics_df.loc[metrics_df["davies_bouldin"].idxmin(), "k"])
        else:
            best_k = k_min_eff

    # Optional stability bias: choose smallest k within 1% of best silhouette
    best_row = metrics_df.loc[metrics_df["k"] == best_k]
    if not best_row.empty and np.isfinite(best_row["silhouette"].iloc[0]):
        best_val = best_row["silhouette"].iloc[0]
        near = metrics_df[np.isfinite(metrics_df["silhouette"]) & (metrics_df["silhouette"] >= 0.99 * best_val)]
        if len(near) > 0:
            best_k = int(near["k"].min())

    return best_k, metrics_df


def fit_cluster_model(train_df, features, *,
                      k_min=2, k_max=12, n_init=20, random_state=42,
                      add_onehot=True, add_distances=True,
                      prefix="cluster"):
    """
    Fit scaler + choose k + fit KMeans on TRAIN ONLY.
    Returns dict: model bundle + train artifacts.
    """
    features = safe_cols(train_df, features)
    if len(features) == 0:
        raise ValueError(f"[{prefix}] No clustering features found in train_df.")

    X_train = _ensure_numeric_matrix(train_df, features)
    scaler = RobustScaler()
    Xs_train = scaler.fit_transform(X_train)

    best_k, metrics_df = choose_k_kmeans(
        Xs_train, k_min=k_min, k_max=k_max, n_init=n_init, random_state=random_state
    )

    if best_k == 1:
        # Degenerate case: can't cluster; everyone same label
        kmeans = None
        train_labels = np.zeros(len(train_df), dtype=int)
        train_dists = np.zeros((len(train_df), 1), dtype=np.float32)
    else:
        kmeans = KMeans(n_clusters=best_k, n_init=n_init, random_state=random_state)
        train_labels = kmeans.fit_predict(Xs_train)

        if add_distances:
            train_dists = kmeans.transform(Xs_train).astype(np.float32)
        else:
            train_dists = None

    bundle = {
        "features": features,
        "scaler": scaler,
        "k": int(best_k),
        "kmeans": kmeans,
        "metrics": metrics_df,
        "add_onehot": add_onehot,
        "add_distances": add_distances,
        "prefix": prefix,
    }
    return bundle, train_labels, train_dists


def apply_cluster_model(df, bundle):
    """
    Apply fitted clustering bundle to any df (train or test).
    Adds:
      - {prefix}_id (int cluster label)
      - one-hot columns (optional)
      - distance columns (optional)
    Returns: new_df
    """
    features = bundle["features"]
    scaler = bundle["scaler"]
    k = bundle["k"]
    kmeans = bundle["kmeans"]
    prefix = bundle["prefix"]

    add_onehot = bundle["add_onehot"]
    add_distances = bundle["add_distances"]

    X = _ensure_numeric_matrix(df, features)
    Xs = scaler.transform(X)

    out = df.copy()

    if k == 1 or kmeans is None:
        labels = np.zeros(len(df), dtype=int)
        dists = np.zeros((len(df), 1), dtype=np.float32) if add_distances else None
    else:
        labels = kmeans.predict(Xs)
        dists = kmeans.transform(Xs).astype(np.float32) if add_distances else None

    # integer label
    out[f"{prefix}_id"] = labels.astype(int)

    # one-hot cluster membership
    if add_onehot:
        for j in range(k):
            out[f"{prefix}_{j}"] = (out[f"{prefix}_id"] == j).astype(int)

    # distances to centroids (soft membership)
    if add_distances:
        # name columns cluster_dist_0 .. cluster_dist_(k-1)
        dist_cols = [f"{prefix}_dist_{j}" for j in range(dists.shape[1])]
        out[dist_cols] = dists

    return out


def attach_clusters(train_df, test_df, features, *,
                    prefix,
                    k_min=2, k_max=12, n_init=20, random_state=42,
                    add_onehot=True, add_distances=True,
                    verbose=True):
    """
    Fit on train, apply to train & test, return updated dfs + model bundle.
    """
    bundle, train_labels, train_dists = fit_cluster_model(
        train_df, features,
        k_min=k_min, k_max=k_max, n_init=n_init, random_state=random_state,
        add_onehot=add_onehot, add_distances=add_distances,
        prefix=prefix
    )
    train_out = apply_cluster_model(train_df, bundle)
    test_out  = apply_cluster_model(test_df,  bundle)

    if verbose:
        print(f"[{prefix}] chosen k = {bundle['k']}")
        # Print top rows of metrics
        m = bundle["metrics"].copy()
        if len(m) > 0:
            print(m.to_string(index=False))

    return train_out, test_out, bundle


# --- Master (full course) clustering
ml_master_train, ml_master_test, bundle_master = attach_clusters(
    ml_master_train, ml_master_test,
    features=CLUSTERING_FEATURES,
    prefix="cluster_master",
    k_min=2, k_max=12,
    n_init=30, random_state=42,
    add_onehot=True,
    add_distances=True,   # set False if you don't want distance features
    verbose=True
)
cluster_models["master"] = bundle_master

# --- Quarterly clustering
for q in ["Q1", "Q2", "Q3", "Q4"]:
    train_df = globals()[f"ml_{q}_train"]
    test_df  = globals()[f"ml_{q}_test"]

    train_df, test_df, bundle_q = attach_clusters(
        train_df, test_df,
        features=CLUSTER_FEATURES_Q[q],
        prefix=f"cluster_{q}",
        k_min=2, k_max=12,
        n_init=30, random_state=42,
        add_onehot=True,
        add_distances=True,
        verbose=True
    )

    globals()[f"ml_{q}_train"] = train_df
    globals()[f"ml_{q}_test"]  = test_df
    cluster_models[q] = bundle_q


# =============================================================================
# Optional: keep train/test aligned after adding clustering columns
# (Useful if you later add/disable one-hot or distances)
# =============================================================================
def align_columns_full(train_df, test_df, keys, target="target_at_risk"):
    train_cols = list(train_df.columns)
    for c in train_cols:
        if c not in test_df.columns:
            # infer dtype type-ish
            if pd.api.types.is_integer_dtype(train_df[c]):
                test_df[c] = 0
            else:
                test_df[c] = 0.0
    # Keep same order
    keep = train_cols
    return train_df[keep].copy(), test_df[keep].copy()

KEYS = ["id_student", "code_module", "code_presentation"]

ml_master_train, ml_master_test = align_columns_full(ml_master_train, ml_master_test, KEYS)
for q in ["Q1", "Q2", "Q3", "Q4"]:
    globals()[f"ml_{q}_train"], globals()[f"ml_{q}_test"] = align_columns_full(
        globals()[f"ml_{q}_train"], globals()[f"ml_{q}_test"], KEYS
    )

print("✓ Clustering features added + train/test aligned")

In [ ]:
def print_cluster_profiles(train_df, features, cluster_id_col, target_col="target_at_risk"):
    """Print mean feature values per cluster with at-risk rate."""
    feats = [f for f in features if f in train_df.columns]
    profile = (
        train_df.groupby(cluster_id_col)[feats + [target_col]]
        .mean()
        .round(3)
    )
    sizes = train_df.groupby(cluster_id_col).size().rename("n_students")
    profile = profile.join(sizes)
    profile["at_risk_%"] = (profile[target_col] * 100).round(1)
    profile = profile.drop(columns=[target_col])
    print(profile.to_string())
    return profile

print("\n" + "="*60)
print("MASTER CLUSTER PROFILES")
print("="*60)
master_profile = print_cluster_profiles(
    ml_master_train, CLUSTERING_FEATURES, "cluster_master_id")

print("\n" + "="*60)
print("Q1 CLUSTER PROFILES")
print("="*60)
q1_profile = print_cluster_profiles(
    ml_Q1_train, CLUSTER_FEATURES_Q["Q1"], "cluster_Q1_id")

print("\n" + "="*60)
print("Q2 CLUSTER PROFILES")
print("="*60)
q2_profile = print_cluster_profiles(
    ml_Q2_train, CLUSTER_FEATURES_Q["Q2"], "cluster_Q2_id")

print("\n" + "="*60)
print("Q3 CLUSTER PROFILES")
print("="*60)
q3_profile = print_cluster_profiles(
    ml_Q3_train, CLUSTER_FEATURES_Q["Q3"], "cluster_Q3_id")

print("\n" + "="*60)
print("Q4 CLUSTER PROFILES")
print("="*60)
q4_profile = print_cluster_profiles(
    ml_Q4_train, CLUSTER_FEATURES_Q["Q4"], "cluster_Q4_id")

In [ ]:
# -----------------------------
# Small utilities
# -----------------------------
def _safe_cols(df, cols):
    return [c for c in cols if c in df.columns]

def _cluster_metrics_from_models(cluster_models: dict):
    rows = []
    for name, bundle in cluster_models.items():
        m = bundle.get("metrics", None)
        if m is None or len(m) == 0:
            continue
        m = m.copy()
        m["table"] = name
        rows.append(m)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    return out[["table","k","silhouette","calinski_harabasz","davies_bouldin"]]

# -----------------------------
# 1) K selection curves
# -----------------------------
def plot_k_selection_curves(cluster_models, tables=("master","Q1","Q2","Q3","Q4")):
    metrics_df = _cluster_metrics_from_models(cluster_models)
    if metrics_df.empty:
        print("No metrics found in cluster_models (bundle['metrics'] missing).")
        return

    for metric in ["silhouette", "calinski_harabasz", "davies_bouldin"]:
        plt.figure(figsize=(9,5))
        for t in tables:
            d = metrics_df[metrics_df["table"] == t].sort_values("k")
            if d.empty:
                continue
            plt.plot(d["k"], d[metric], marker="o", label=str(t))
        plt.title(f"K selection curves: {metric}")
        plt.xlabel("k")
        plt.ylabel(metric)
        plt.legend(title="Table")
        plt.tight_layout()
        plt.show()

# -----------------------------
# 2) Cluster profile heatmap (means)
# -----------------------------
def cluster_profiles_table(df, cluster_col, feature_cols, outcome_col="target_at_risk"):
    feats = _safe_cols(df, feature_cols)
    base_cols = [cluster_col] + feats + ([outcome_col] if outcome_col in df.columns else [])
    base = df[base_cols].copy()

    for c in feats:
        base[c] = pd.to_numeric(base[c], errors="coerce").fillna(0.0)

    prof = base.groupby(cluster_col)[feats].mean()
    n = base.groupby(cluster_col).size().rename("n_students")
    out = prof.merge(n, left_index=True, right_index=True)

    if outcome_col in base.columns:
        at_risk = base.groupby(cluster_col)[outcome_col].mean().rename("at_risk_%") * 100.0
        out = out.merge(at_risk, left_index=True, right_index=True)

    return out.sort_index()



def plot_cluster_heatmap_with_values(df, cluster_col, feature_cols, title):
    
    profile = (
        df.groupby(cluster_col)[feature_cols]
        .mean()
        .sort_index()
    )

    plt.figure(figsize=(10,5))
    sns.heatmap(
        profile,
        annot=True,          # <-- THIS shows the numbers
        fmt=".2f",           # <-- rounds to 2 decimals (0.78)
        cmap="coolwarm",
        linewidths=0.5
    )

    plt.title(title)
    plt.xlabel("Features")
    plt.ylabel("Cluster")
    plt.tight_layout()
    plt.show()

# -----------------------------
# 3) Safe vs At-Risk bars (side-by-side)
# -----------------------------
def plot_safe_risk_side_by_side(df, cluster_col, title, outcome_col="target_at_risk"):
    dist = (
        df.groupby([cluster_col, outcome_col])
        .size()
        .unstack(fill_value=0)
        .sort_index()
    )
    if 0 not in dist.columns: dist[0] = 0
    if 1 not in dist.columns: dist[1] = 0
    dist = dist[[0,1]]
    dist.columns = ["Safe", "At_Risk"]

    clusters = dist.index.astype(str).tolist()
    x = np.arange(len(clusters))
    w = 0.38

    plt.figure(figsize=(9,5))
    plt.bar(x - w/2, dist["Safe"].to_numpy(), w, label="Safe")
    plt.bar(x + w/2, dist["At_Risk"].to_numpy(), w, label="At Risk")
    plt.xticks(x, clusters)
    plt.xlabel("Cluster")
    plt.ylabel("Number of students")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_safe_risk_side_by_side_pct(df, cluster_col, title, outcome_col="target_at_risk"):
    dist = (
        df.groupby([cluster_col, outcome_col])
        .size()
        .unstack(fill_value=0)
        .sort_index()
    )
    if 0 not in dist.columns: dist[0] = 0
    if 1 not in dist.columns: dist[1] = 0
    dist = dist[[0,1]]
    dist = dist.div(dist.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0) * 100.0
    dist.columns = ["Safe_%", "At_Risk_%"]

    clusters = dist.index.astype(str).tolist()
    x = np.arange(len(clusters))
    w = 0.38

    plt.figure(figsize=(9,5))
    plt.bar(x - w/2, dist["Safe_%"].to_numpy(), w, label="Safe %")
    plt.bar(x + w/2, dist["At_Risk_%"].to_numpy(), w, label="At Risk %")
    plt.xticks(x, clusters)
    plt.xlabel("Cluster")
    plt.ylabel("Percentage of students")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

# -----------------------------
# 4) At-risk rate bar chart
# -----------------------------
def plot_risk_rate_by_cluster(df, cluster_col, title, outcome_col="target_at_risk"):
    rate = df.groupby(cluster_col)[outcome_col].mean().sort_index() * 100.0
    clusters = rate.index.astype(str).tolist()
    x = np.arange(len(clusters))

    plt.figure(figsize=(9,5))
    plt.bar(x, rate.to_numpy())
    plt.xticks(x, clusters)
    plt.xlabel("Cluster")
    plt.ylabel("At-risk rate (%)")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# -----------------------------
# 5) One function: run EVERYTHING (Master + Q1–Q4)
# -----------------------------
def visualize_all(ml_master_train, ml_Q1_train, ml_Q2_train, ml_Q3_train, ml_Q4_train,
                  CLUSTERING_FEATURES, CLUSTER_FEATURES_Q,
                  cluster_models=None):

    # K selection curves (optional)
    if cluster_models is not None:
        plot_k_selection_curves(cluster_models, tables=("master","Q1","Q2","Q3","Q4"))

    # # Master
    # plot_cluster_profile_heatmap(
    #     ml_master_train, "cluster_master_id", CLUSTERING_FEATURES,
    #     "Master: cluster feature means (train)"
    # )
    plot_safe_risk_side_by_side(
        ml_master_train, "cluster_master_id",
        "Master: Safe vs At-Risk (counts, train)"
    )
    plot_safe_risk_side_by_side_pct(
        ml_master_train, "cluster_master_id",
        "Master: Safe vs At-Risk (percent, train)"
    )
    plot_risk_rate_by_cluster(
        ml_master_train, "cluster_master_id",
        "Master: At-risk rate by cluster (train)"
    )

    # Quarters
    quarter_map = {
        "Q1": (ml_Q1_train, "cluster_Q1_id"),
        "Q2": (ml_Q2_train, "cluster_Q2_id"),
        "Q3": (ml_Q3_train, "cluster_Q3_id"),
        "Q4": (ml_Q4_train, "cluster_Q4_id"),
    }
    for q, (df, ccol) in quarter_map.items():
        # plot_cluster_profile_heatmap(
        #     df, ccol, CLUSTER_FEATURES_Q[q],
        #     f"{q}: cluster feature means (train)"
        # )
        plot_safe_risk_side_by_side(
            df, ccol,
            f"{q}: Safe vs At-Risk (counts, train)"
        )
        plot_safe_risk_side_by_side_pct(
            df, ccol,
            f"{q}: Safe vs At-Risk (percent, train)"
        )
        plot_risk_rate_by_cluster(
            df, ccol,
            f"{q}: At-risk rate by cluster (train)"
        )


visualize_all(
    ml_master_train, ml_Q1_train, ml_Q2_train, ml_Q3_train, ml_Q4_train,
    CLUSTERING_FEATURES, CLUSTER_FEATURES_Q,
    cluster_models=cluster_models   # optional; remove if you don’t want k-curves
)

In [ ]:
def plot_cluster_heatmap(df, cluster_col, feature_cols, title):
    profile = (
        df.groupby(cluster_col)[feature_cols]
        .mean()
    )

    plt.figure(figsize=(10,5))
    sns.heatmap(profile, annot=True, fmt=".2f", cmap="coolwarm")
    plt.title(title)
    plt.xlabel("Features")
    plt.ylabel("Cluster")
    plt.tight_layout()
    plt.show()

plot_cluster_heatmap(
    df=ml_master_train,
    cluster_col="cluster_master_id",
    feature_cols=CLUSTERING_FEATURES,
    title="Master Cluster Feature Profiles"
)

plot_cluster_heatmap(
    ml_Q1_train,
    cluster_col="cluster_Q1_id",
    feature_cols=CLUSTER_FEATURES_Q["Q1"],
    title="Q1 Cluster Feature Profiles"
)

plot_cluster_heatmap(
    ml_Q2_train,
    cluster_col="cluster_Q2_id",
    feature_cols=CLUSTER_FEATURES_Q["Q2"],
    title="Q2 Cluster Feature Profiles"
)

plot_cluster_heatmap(
    ml_Q3_train,
    cluster_col="cluster_Q3_id",
    feature_cols=CLUSTER_FEATURES_Q["Q3"],
    title="Q3 Cluster Feature Profiles"
)

plot_cluster_heatmap(
    ml_Q4_train,
    cluster_col="cluster_Q4_id",
    feature_cols=CLUSTER_FEATURES_Q["Q4"],
    title="Q4 Cluster Feature Profiles"
)

## ML Model

In [ ]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
warnings.filterwarnings("ignore")

from scipy.stats import pointbiserialr
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, cross_validate
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
torch.set_num_threads(1)
torch.set_num_interop_threads(1)  # also add this
sns.set_style("whitegrid")
plt.rcParams.update({"figure.facecolor": "white",
                     "axes.spines.top": False,
                     "axes.spines.right": False})

KEYS   = ["id_student", "code_module", "code_presentation"]
TARGET = "target_at_risk"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Libraries loaded.")
print("Device:", device)
print("\nClass balance (train):")
print(ml_master_train[TARGET].value_counts())
print(ml_master_train[TARGET].value_counts(normalize=True).round(3))

In [ ]:
# =============================================================================
# CELL 2 — Feature columns per window
# Reads directly from your already-built ml_* tables
# =============================================================================

def get_feature_cols(df):
    return [c for c in df.columns if c not in KEYS + [TARGET]]


# Your tables are already split — just read the feature lists
WINDOWS = {
    "Q1": {
        "train": ml_Q1_train,
        "test":  ml_Q1_test,
        "features": get_feature_cols(ml_Q1_train),
    },
    "Q2": {
        "train": ml_Q2_train,
        "test":  ml_Q2_test,
        "features": get_feature_cols(ml_Q2_train),
    },
    "Q3": {
        "train": ml_Q3_train,
        "test":  ml_Q3_test,
        "features": get_feature_cols(ml_Q3_train),
    },
    "Master": {
        "train": ml_master_train,
        "test":  ml_master_test,
        "features": get_feature_cols(ml_master_train),
    },
}

print("Windows:")
for label, w in WINDOWS.items():
    print(f"  {label:10s}  "
          f"train={len(w['train']):,}  "
          f"test={len(w['test']):,}  "
          f"features={len(w['features'])}")
          
print("\nTest presentations (held out per module):")
print(test_presentations.to_string(index=False))

In [ ]:
# =============================================================================
# CELL 3 — Feature Selection
# Fit ONLY on train — test never seen during selection
# =============================================================================

# def correlation_filter(X_df, features, target_series, threshold=0.85):
#     """Remove features with pairwise correlation > threshold."""
#     rows = []
#     for f in features:
#         corr, pval = pointbiserialr(target_series.astype(int), X_df[f])
#         rows.append({"Feature": f,
#                      "Correlation": round(corr, 4),
#                      "P-Value": round(pval, 6)})
#     corr_target = (pd.DataFrame(rows)
#                    .sort_values("Correlation", key=abs, ascending=False)
#                    .set_index("Feature"))

#     corr_matrix = X_df[features].corr().abs()
#     upper = corr_matrix.where(
#         np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
#     to_drop = [col for col in upper.columns
#                if any(upper[col] > threshold)]
#     kept = [f for f in features if f not in to_drop]
#     print(f"    Correlation: {len(features)} -> {len(kept)} "
#           f"(dropped: {to_drop})")
#     return kept, corr_target


# SELECTED_FEATURES = {}
# SELECTION_SUMMARY = []

# for window_label, w in WINDOWS.items():
#     print(f"\n{'='*55}")
#     print(f"  Feature selection — {window_label}")
#     print(f"{'='*55}")

#     train_df = w["train"]
#     features = w["features"]

#     # Use train only — test never touched here
#     X_train      = train_df[features].fillna(0)
#     y_train      = train_df[TARGET].astype(int)

#     after_corr, corr_scores = correlation_filter(
#         X_train, features, y_train)

#     SELECTED_FEATURES[window_label] = after_corr
#     SELECTION_SUMMARY.append({
#         "Window":   window_label,
#         "Original": len(features),
#         "Final":    len(after_corr),
#         "Selected": after_corr,
#     })
#     print(f"    Final ({len(after_corr)}): {after_corr}")

# summary_df = pd.DataFrame(SELECTION_SUMMARY)
# print("\n" + "="*55)
# print("FEATURE SELECTION SUMMARY")
# print("="*55)
# print(summary_df[["Window","Original","Final"]].to_string(index=False))

def correlation_filter(X_df, features, threshold=0.85, verbose=True):
    """
    Unsupervised correlation filter:
    - Computes pairwise feature-feature correlation
    - Drops one feature from highly correlated pairs
    - Does NOT use target association

    This is more appropriate when correlation analysis is being used
    as a multicollinearity / redundancy check rather than supervised
    feature ranking.
    """
    X = X_df[features].copy()

    # absolute correlation matrix
    corr_matrix = X.corr().abs()

    # upper triangle only
    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    # collect pairs above threshold
    high_corr_pairs = []
    to_drop = set()

    for col in upper.columns:
        correlated_with_col = upper.index[upper[col] > threshold].tolist()
        for row in correlated_with_col:
            high_corr_pairs.append((row, col, upper.loc[row, col]))
            # simple rule: drop the later column
            to_drop.add(col)

    kept = [f for f in features if f not in to_drop]

    if verbose:
        print(f"    Correlation filter: {len(features)} -> {len(kept)}")
        if high_corr_pairs:
            print("    Highly correlated pairs:")
            for f1, f2, corr_val in sorted(high_corr_pairs, key=lambda x: -x[2]):
                print(f"      {f1} vs {f2} : r = {corr_val:.3f}")
        else:
            print("    No highly correlated pairs found.")

        if to_drop:
            print(f"    Dropped features: {sorted(to_drop)}")

    return kept, corr_matrix, high_corr_pairs

In [ ]:
SELECTED_FEATURES = {}
SELECTION_SUMMARY = []
CORR_INFO = {}

for window_label, w in WINDOWS.items():
    print(f"\n{'='*55}")
    print(f"  Feature selection — {window_label}")
    print(f"{'='*55}")

    train_df = w["train"]
    features = w["features"]

    # train only
    X_train = train_df[features].fillna(0)

    kept, corr_matrix, high_corr_pairs = correlation_filter(
        X_train,
        features,
        threshold=0.85,
        verbose=True
    )

    SELECTED_FEATURES[window_label] = kept
    CORR_INFO[window_label] = {
        "corr_matrix": corr_matrix,
        "high_corr_pairs": high_corr_pairs
    }

    SELECTION_SUMMARY.append({
        "Window": window_label,
        "Original": len(features),
        "Final": len(kept),
        "Dropped": sorted(list(set(features) - set(kept))),
        "Selected": kept,
    })

summary_df = pd.DataFrame(SELECTION_SUMMARY)

print("\n" + "="*55)
print("FEATURE SELECTION SUMMARY")
print("="*55)
print(summary_df[["Window", "Original", "Final"]].to_string(index=False))

In [ ]:
# =============================================================================
# CELL 4 — Models
# =============================================================================

from sklearn.preprocessing import RobustScaler   # add this to your imports

def make_models(spw=1.0):
    return {
        "Logistic Regression": Pipeline([
            ("scaler", RobustScaler()),            # was StandardScaler()
            ("clf", LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            ))
        ]),
        "SVM": Pipeline([
            ("scaler", RobustScaler()),            # was StandardScaler()
            ("clf", SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=True,
                C=1.0,
                random_state=42
            ))
        ]),
        "Random Forest": Pipeline([
            ("scaler", RobustScaler()),            # was StandardScaler()
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=10,
                min_samples_split=20,
                max_features="sqrt",
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            ))
        ]),
        "XGBoost": Pipeline([
    ("scaler", RobustScaler()),
    ("clf", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,           # was 4 — reduce complexity
        subsample=0.7,         # was 0.8 — more aggressive subsampling
        colsample_bytree=0.7,  # was 0.8 — more aggressive
        reg_alpha=0.5,         # was 0.1 — stronger L1
        reg_lambda=2.0,        # was 1.5 — stronger L2
        scale_pos_weight=spw,
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    ))
]),
    }


def score_row(name, model, X_te, y_te, window_label, features, y_prob=None):
    y_pred       = model.predict(X_te)
    if y_prob is None:
        y_prob   = model.predict_proba(X_te)[:, 1]
    row = {
        "Window":    window_label,
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_te, y_pred),                  4),
        "Precision": round(precision_score(y_te, y_pred, zero_division=0), 4),
        "Recall":    round(recall_score(y_te, y_pred, zero_division=0),    4),
        "F1":        round(f1_score(y_te, y_pred, zero_division=0),        4),
        "ROC-AUC":   round(roc_auc_score(y_te, y_prob),                   4),
        "_model":    model,
        "_X_te":     X_te,
        "_y_te":     y_te,
        "_features": features,
        "_y_prob":   y_prob,
    }
    print(f"    {name:22s}  "
          f"Acc={row['Accuracy']:.3f}  "
          f"Prec={row['Precision']:.3f}  "
          f"Rec={row['Recall']:.3f}  "
          f"F1={row['F1']:.3f}  "
          f"AUC={row['ROC-AUC']:.3f}")
    return row


print("Models defined.")

In [ ]:
# # =============================================================================
# # CELL 5 — Cross Validation  (fixed: GroupKFold by module presentation)
# # =============================================================================

# from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_validate

# def run_cross_validation(window_label, X_tr, y_tr, spw, groups=None):
#     """
#     5-fold CV on train data.
#     If groups provided → GroupKFold by presentation (more conservative).
#     Otherwise → StratifiedKFold.
#     """
#     if groups is not None:
#         cv = GroupKFold(n_splits=3)
#         cv_kwargs = {"groups": groups}
#     else:
#         cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#         cv_kwargs = {}

#     models  = make_models(spw=spw)
#     cv_rows = []

#     print(f"\n    --- 5-Fold CV on train cohort (GroupKFold by presentation) ---")
#     for name, model in models.items():
#         cv_results = cross_validate(
#             model, X_tr, y_tr,
#             cv=cv,
#             groups=groups,
#             scoring={
#                 "f1":      "f1",
#                 "roc_auc": "roc_auc",
#                 "recall":  "recall",
#             },
#             return_train_score=True,
#             n_jobs=-1
#         )
#         row = {
#             "Window":      window_label,
#             "Model":       name,
#             "CV F1":       round(cv_results["test_f1"].mean(),       4),
#             "CV F1 Std":   round(cv_results["test_f1"].std(),        4),
#             "CV AUC":      round(cv_results["test_roc_auc"].mean(),  4),
#             "CV AUC Std":  round(cv_results["test_roc_auc"].std(),   4),
#             "CV Recall":   round(cv_results["test_recall"].mean(),   4),
#             "Train AUC":   round(cv_results["train_roc_auc"].mean(), 4),
#             "Overfit Gap": round(
#                 cv_results["train_roc_auc"].mean() -
#                 cv_results["test_roc_auc"].mean(), 4),
#         }
#         cv_rows.append(row)
#         status = "OK" if row["Overfit Gap"] < 0.05 else "OVERFIT"
#         print(f"    {name:22s}  "
#               f"CV_AUC={row['CV AUC']:.3f}"
#               f"(+/-{row['CV AUC Std']:.3f})  "
#               f"CV_F1={row['CV F1']:.3f}  "
#               f"Gap={row['Overfit Gap']:.3f}  {status}")

#     return pd.DataFrame(cv_rows)

# print("Cross validation defined.")
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_validate

def run_cross_validation(window_label, X_tr, y_tr, spw, groups=None):
    """
    CV on train data.
    If groups provided -> GroupKFold by module-presentation.
    Otherwise -> StratifiedKFold.
    """
    if groups is not None:
        unique_groups = pd.Series(groups).nunique()
        n_splits = min(3, unique_groups)

        if n_splits < 2:
            raise ValueError(
                f"Not enough unique groups for GroupKFold in {window_label}. "
                f"Found only {unique_groups} unique groups."
            )

        cv = GroupKFold(n_splits=n_splits)
        print(f"\n    --- GroupKFold CV on train cohort ({n_splits} folds, module-presentation groups) ---")
    else:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        print(f"\n    --- Stratified 5-Fold CV on train cohort ---")

    models = make_models(spw=spw)
    cv_rows = []

    for name, model in models.items():
        cv_results = cross_validate(
            model,
            X_tr,
            y_tr,
            cv=cv,
            groups=groups,
            scoring={
                "f1": "f1",
                "roc_auc": "roc_auc",
                "recall": "recall",
            },
            return_train_score=True,
            n_jobs=-1
        )

        row = {
            "Window": window_label,
            "Model": name,
            "CV F1": round(cv_results["test_f1"].mean(), 4),
            "CV F1 Std": round(cv_results["test_f1"].std(), 4),
            "CV AUC": round(cv_results["test_roc_auc"].mean(), 4),
            "CV AUC Std": round(cv_results["test_roc_auc"].std(), 4),
            "CV Recall": round(cv_results["test_recall"].mean(), 4),
            "Train AUC": round(cv_results["train_roc_auc"].mean(), 4),
            "Overfit Gap": round(
                cv_results["train_roc_auc"].mean() - cv_results["test_roc_auc"].mean(), 4
            ),
        }
        cv_rows.append(row)

        status = "OK" if row["Overfit Gap"] < 0.05 else "OVERFIT"
        print(
            f"    {name:22s}  "
            f"CV_AUC={row['CV AUC']:.3f}(+/-{row['CV AUC Std']:.3f})  "
            f"CV_F1={row['CV F1']:.3f}  "
            f"Gap={row['Overfit Gap']:.3f}  {status}"
        )

    return pd.DataFrame(cv_rows)

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
import numpy as np
import pandas as pd

def find_best_threshold(y_true, y_prob, metric="f1"):
    """
    Select threshold on TRAIN only.
    Default metric = F1.
    """
    thresholds = np.arange(0.10, 0.91, 0.02)
    best_thr = 0.50
    best_score = -1

    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)

        if metric == "f1":
            score = f1_score(y_true, y_pred, zero_division=0)
        elif metric == "recall":
            score = recall_score(y_true, y_pred, zero_division=0)
        elif metric == "precision":
            score = precision_score(y_true, y_pred, zero_division=0)
        else:
            raise ValueError("metric must be one of: f1, recall, precision")

        if score > best_score:
            best_score = score
            best_thr = thr

    return best_thr, best_score


def fit_model_and_get_probs(model, X_tr, y_tr, X_te):
    """
    Fit model on train and return train/test probabilities.
    """
    model.fit(X_tr, y_tr)
    y_prob_tr = model.predict_proba(X_tr)[:, 1]
    y_prob_te = model.predict_proba(X_te)[:, 1]
    return model, y_prob_tr, y_prob_te


def score_probabilities(name, y_te, y_prob_te, window_label, features,
                        model_obj=None, X_te=None, threshold=0.5):
    """
    Score test predictions using a chosen threshold.
    """
    y_pred = (y_prob_te >= threshold).astype(int)

    row = {
        "Window": window_label,
        "Model": name,
        "Threshold": round(float(threshold), 3),
        "Accuracy": round(accuracy_score(y_te, y_pred), 4),
        "Precision": round(precision_score(y_te, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_te, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_te, y_pred, zero_division=0), 4),
        "ROC-AUC": round(roc_auc_score(y_te, y_prob_te), 4),
        "_model": model_obj,
        "_X_te": X_te,
        "_y_te": y_te,
        "_features": features,
        "_y_prob": y_prob_te,
    }

    print(
        f"    {name:22s}  "
        f"Thr={row['Threshold']:.2f}  "
        f"Acc={row['Accuracy']:.3f}  "
        f"Prec={row['Precision']:.3f}  "
        f"Rec={row['Recall']:.3f}  "
        f"F1={row['F1']:.3f}  "
        f"AUC={row['ROC-AUC']:.3f}"
    )
    return row

In [ ]:
# =============================================================================
# CELL 6 — MLP   (fixed: logits not probs, seeds added)
# =============================================================================

# ── Reproducibility seeds ────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
            # ── No Sigmoid here — BCEWithLogitsLoss expects raw logits ──────
        )

    def forward(self, x):
        return self.net(x).squeeze(1)   # returns logits


# def train_mlp(X_tr, y_tr, X_te, y_te,
#               epochs=60, batch_size=256, lr=1e-3):
#     torch.manual_seed(42)
#     np.random.seed(42)

#     scaler  = RobustScaler()
#     X_tr_s  = scaler.fit_transform(X_tr)
#     X_te_s  = scaler.transform(X_te)

#     pos_weight = torch.tensor(
#         [(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)],
#         dtype=torch.float32).to(device)

#     loader = DataLoader(
#         TensorDataset(
#             torch.tensor(X_tr_s, dtype=torch.float32),
#             torch.tensor(y_tr,   dtype=torch.float32)
#         ),
#         batch_size=batch_size, shuffle=True
#     )

#     model     = MLP(X_tr.shape[1]).to(device)
#     criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
#     optimizer = optim.Adam(
#         model.parameters(), lr=lr, weight_decay=1e-4)
#     scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer, patience=5, factor=0.5)

#     for epoch in range(epochs):
#         model.train()
#         epoch_loss = 0
#         for xb, yb in loader:
#             xb, yb = xb.to(device), yb.to(device)
#             optimizer.zero_grad()
#             logits = model(xb)                    # raw logits
#             loss   = criterion(logits, yb)        # BCEWithLogitsLoss on logits
#             loss.backward()
#             optimizer.step()
#             epoch_loss += loss.item()
#         scheduler.step(epoch_loss)
#         if (epoch + 1) % 20 == 0:
#             print(f"      Epoch {epoch+1}/{epochs}  "
#                   f"loss={epoch_loss/len(loader):.4f}")

#     model.eval()
#     with torch.no_grad():
#         logits = model(
#             torch.tensor(X_te_s, dtype=torch.float32).to(device)
#         )
#         y_prob = torch.sigmoid(logits).cpu().numpy()   # sigmoid only at inference

#     return y_prob, (y_prob >= 0.5).astype(int)


# print("MLP defined.")
from sklearn.model_selection import train_test_split

# def train_mlp(X_tr, y_tr, X_te, y_te,
#               epochs=60, batch_size=512, lr=1e-3, patience=10):
#     torch.manual_seed(42)
#     np.random.seed(42)

#     X_tr_inner, X_val, y_tr_inner, y_val = train_test_split(
#         X_tr, y_tr,
#         test_size=0.15,
#         random_state=42,
#         stratify=y_tr
#     )

#     scaler = RobustScaler()
#     X_tr_s = scaler.fit_transform(X_tr_inner)
#     X_val_s = scaler.transform(X_val)
#     X_te_s = scaler.transform(X_te)

#     pos_weight = torch.tensor(
#         [(y_tr_inner == 0).sum() / max((y_tr_inner == 1).sum(), 1)],
#         dtype=torch.float32
#     ).to(device)

#     train_loader = DataLoader(
#         TensorDataset(
#             torch.tensor(X_tr_s, dtype=torch.float32),
#             torch.tensor(y_tr_inner, dtype=torch.float32)
#         ),
#         batch_size=batch_size,
#         shuffle=True
#     )

#     model = MLP(X_tr.shape[1]).to(device)
#     criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
#     optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

#     best_state = None
#     best_val_auc = -np.inf
#     wait = 0

#     X_val_t = torch.tensor(X_val_s, dtype=torch.float32).to(device)

#     for epoch in range(epochs):
#         model.train()
#         epoch_loss = 0.0

#         for xb, yb in train_loader:
#             xb, yb = xb.to(device), yb.to(device)
#             optimizer.zero_grad()
#             logits = model(xb)
#             loss = criterion(logits, yb)
#             loss.backward()
#             optimizer.step()
#             epoch_loss += loss.item()

#         model.eval()
#         with torch.no_grad():
#             val_logits = model(X_val_t)
#             val_prob = torch.sigmoid(val_logits).cpu().numpy()

#         val_auc = roc_auc_score(y_val, val_prob)

#         if val_auc > best_val_auc:
#             best_val_auc = val_auc
#             best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
#             wait = 0
#         else:
#             wait += 1

#         if (epoch + 1) % 20 == 0:
#             print(f"      Epoch {epoch+1}/{epochs}  loss={epoch_loss/len(train_loader):.4f}  val_auc={val_auc:.4f}")

#         if wait >= patience:
#             print(f"      Early stopping at epoch {epoch+1}")
#             break

#     model.load_state_dict(best_state)
#     model.eval()

#     with torch.no_grad():
#         tr_logits = model(torch.tensor(scaler.transform(X_tr), dtype=torch.float32).to(device))
#         te_logits = model(torch.tensor(X_te_s, dtype=torch.float32).to(device))

#         y_prob_tr = torch.sigmoid(tr_logits).cpu().numpy()
#         y_prob_te = torch.sigmoid(te_logits).cpu().numpy()

#     return y_prob_tr, y_prob_te
def train_mlp(X_tr, y_tr, X_te, y_te,
              epochs=40, batch_size=1024, lr=3e-3, patience=5):
    torch.manual_seed(42)
    np.random.seed(42)

    print(f"      [MLP] Input shape: {X_tr.shape} | pos={y_tr.sum()} neg={(y_tr==0).sum()}")

    # ── Split ────────────────────────────────────────────────────────────────
    print(f"      [MLP] Splitting train/val...")
    X_tr_inner, X_val, y_tr_inner, y_val = train_test_split(
        X_tr, y_tr, test_size=0.10, random_state=42, stratify=y_tr
    )
    print(f"      [MLP] Train inner: {X_tr_inner.shape} | Val: {X_val.shape}")

    # ── Scaling ──────────────────────────────────────────────────────────────
    print(f"      [MLP] Scaling...")
    scaler = RobustScaler()
    X_tr_s = scaler.fit_transform(X_tr_inner)
    X_val_s = scaler.transform(X_val)
    X_te_s  = scaler.transform(X_te)

    pos_weight = torch.tensor(
        [(y_tr_inner == 0).sum() / max((y_tr_inner == 1).sum(), 1)],
        dtype=torch.float32
    ).to(device)
    print(f"      [MLP] pos_weight: {pos_weight.item():.3f}")

    # ── DataLoader ───────────────────────────────────────────────────────────
    print(f"      [MLP] Building DataLoader (batch_size={batch_size})...")
    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_tr_s, dtype=torch.float32),
            torch.tensor(y_tr_inner, dtype=torch.float32)
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,    # ← add this — prevents subprocess spawning which can hang

    )
    print(f"      [MLP] Batches per epoch: {len(train_loader)}")

    # ── Model init ───────────────────────────────────────────────────────────
    model = MLP(X_tr.shape[1]).to(device)
    print(f"      [MLP] Model on device: {device}")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_state   = None
    best_val_auc = -np.inf
    wait         = 0
    X_val_t      = torch.tensor(X_val_s, dtype=torch.float32).to(device)

    print(f"      [MLP] Starting training loop ({epochs} max epochs, patience={patience})...")

    # ── Training loop ────────────────────────────────────────────────────────
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_prob   = torch.sigmoid(val_logits).cpu().numpy()

        val_auc = roc_auc_score(y_val, val_prob)

        # Print every epoch so you can see exactly where it stalls
        print(f"      [MLP] Epoch {epoch+1:>3}/{epochs}  "
              f"loss={epoch_loss/len(train_loader):.4f}  "
              f"val_auc={val_auc:.4f}  "
              f"wait={wait}/{patience}")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait         = 0
        else:
            wait += 1

        if wait >= patience:
            print(f"      [MLP] Early stopping at epoch {epoch+1} "
                  f"(best val_auc={best_val_auc:.4f})")
            break

    # ── Final inference ──────────────────────────────────────────────────────
    print(f"      [MLP] Training done. Running final inference...")
    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        tr_logits = model(torch.tensor(
            scaler.transform(X_tr), dtype=torch.float32).to(device))
        te_logits = model(torch.tensor(
            X_te_s, dtype=torch.float32).to(device))

        y_prob_tr = torch.sigmoid(tr_logits).cpu().numpy()
        y_prob_te = torch.sigmoid(te_logits).cpu().numpy()

    print(f"      [MLP] Done ✓")
    return y_prob_tr, y_prob_te

In [ ]:
# =============================================================================
# LSTM + BiLSTM  (fixed: logits, seeds, masking per-student quarter cutoffs)
# =============================================================================

torch.manual_seed(42)
np.random.seed(42)

class LSTMModel(nn.Module):
    def __init__(self, input_size=3, hidden_size=64,
                 num_layers=2, dropout=0.3, bidirectional=False):
        super().__init__()
        self.bidirectional = bidirectional
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        factor = 2 if bidirectional else 1
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * factor, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
            # ── No Sigmoid — BCEWithLogitsLoss expects raw logits ────────────
        )

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        last = (torch.cat([h_n[-2], h_n[-1]], dim=1)
                if self.bidirectional else h_n[-1])
        return self.classifier(last).squeeze(1)   # returns logits


def build_sequences_masked(df, global_max_week, q_end_col):
    """
    Build (n_students, global_max_week, 3) sequences.
    Weeks beyond each student's true quarter cutoff are masked to 0.
    This keeps fixed shape while staying faithful to per-course quarter windows.
    
    df         : ml_seq_train or ml_seq_test (has wkXX_* cols + q_end_col)
    global_max_week : max week to include (e.g. 10 for Q1)
    q_end_col  : column in df with per-student true quarter end week
    """
    pat   = re.compile(r"^wk(\d{2})_(content|assessment|social)_norm$")
    weeks = sorted({int(pat.match(c).group(1))
                    for c in df.columns if pat.match(c)
                    and int(pat.match(c).group(1)) <= global_max_week})
    if not weeks:
        return None

    n = len(df)
    seq = np.zeros((n, len(weeks), 3), dtype=np.float32)

    true_cutoffs = df[q_end_col].values if q_end_col in df.columns \
                   else np.full(n, global_max_week)

    for wi, wk in enumerate(weeks):
        c = df.get(f"wk{wk:02d}_content_norm",
                   pd.Series(0.0, index=df.index)).fillna(0).values
        a = df.get(f"wk{wk:02d}_assessment_norm",
                   pd.Series(0.0, index=df.index)).fillna(0).values
        s = df.get(f"wk{wk:02d}_social_norm",
                   pd.Series(0.0, index=df.index)).fillna(0).values

        # mask weeks beyond each student's true quarter cutoff
        mask = (wk <= true_cutoffs).astype(np.float32)
        seq[:, wi, 0] = c * mask
        seq[:, wi, 1] = a * mask
        seq[:, wi, 2] = s * mask

    return seq   # (n_students, n_weeks, 3)


# def train_seq_model(X_tr, y_tr, X_te, y_te,
#                     bidirectional=False,
#                     epochs=60, batch_size=256, lr=1e-3):
#     torch.manual_seed(42)
#     np.random.seed(42)

#     # per-feature normalisation on train only
#     mean   = X_tr.mean(axis=(0, 1), keepdims=True)
#     std    = X_tr.std(axis=(0, 1),  keepdims=True) + 1e-8
#     X_tr_s = (X_tr - mean) / std
#     X_te_s = (X_te - mean) / std

#     pos_weight = torch.tensor(
#         [(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)],
#         dtype=torch.float32).to(device)

#     loader = DataLoader(
#         TensorDataset(
#             torch.tensor(X_tr_s, dtype=torch.float32),
#             torch.tensor(y_tr,   dtype=torch.float32)
#         ),
#         batch_size=batch_size, shuffle=True
#     )

#     model     = LSTMModel(bidirectional=bidirectional).to(device)
#     criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
#     optimizer = optim.Adam(
#         model.parameters(), lr=lr, weight_decay=1e-4)
#     scheduler = optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer, patience=5, factor=0.5)

#     for epoch in range(epochs):
#         model.train()
#         epoch_loss = 0
#         for xb, yb in loader:
#             xb, yb = xb.to(device), yb.to(device)
#             optimizer.zero_grad()
#             logits = model(xb)                    # raw logits
#             loss   = criterion(logits, yb)        # correct usage
#             loss.backward()
#             nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#             optimizer.step()
#             epoch_loss += loss.item()
#         scheduler.step(epoch_loss)
#         if (epoch + 1) % 20 == 0:
#             print(f"      Epoch {epoch+1}/{epochs}  "
#                   f"loss={epoch_loss/len(loader):.4f}")

#     model.eval()
#     with torch.no_grad():
#         logits = model(
#             torch.tensor(X_te_s, dtype=torch.float32).to(device)
#         )
#         y_prob = torch.sigmoid(logits).cpu().numpy()  # sigmoid only at inference

#     return y_prob, (y_prob >= 0.5).astype(int)

def train_seq_model(X_tr, y_tr, X_te, y_te,
                    bidirectional=False,
                    epochs=60, batch_size=512, lr=1e-3, patience=10):
    torch.manual_seed(42)
    np.random.seed(42)

    idx = np.arange(len(y_tr))
    tr_idx, val_idx = train_test_split(
        idx,
        test_size=0.15,
        random_state=42,
        stratify=y_tr
    )

    X_tr_inner, y_tr_inner = X_tr[tr_idx], y_tr[tr_idx]
    X_val, y_val = X_tr[val_idx], y_tr[val_idx]

    mean = X_tr_inner.mean(axis=(0, 1), keepdims=True)
    std = X_tr_inner.std(axis=(0, 1), keepdims=True) + 1e-8

    X_tr_s = (X_tr_inner - mean) / std
    X_val_s = (X_val - mean) / std
    X_te_s = (X_te - mean) / std
    X_tr_full_s = (X_tr - mean) / std

    pos_weight = torch.tensor(
        [(y_tr_inner == 0).sum() / max((y_tr_inner == 1).sum(), 1)],
        dtype=torch.float32
    ).to(device)

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_tr_s, dtype=torch.float32),
            torch.tensor(y_tr_inner, dtype=torch.float32)
        ),
        batch_size=batch_size,
        shuffle=True
    )

    model = LSTMModel(bidirectional=bidirectional).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_state = None
    best_val_auc = -np.inf
    wait = 0

    X_val_t = torch.tensor(X_val_s, dtype=torch.float32).to(device)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_prob = torch.sigmoid(val_logits).cpu().numpy()

        val_auc = roc_auc_score(y_val, val_prob)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if (epoch + 1) % 20 == 0:
            print(f"      Epoch {epoch+1}/{epochs}  loss={epoch_loss/len(train_loader):.4f}  val_auc={val_auc:.4f}")

        if wait >= patience:
            print(f"      Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        tr_logits = model(torch.tensor(X_tr_full_s, dtype=torch.float32).to(device))
        te_logits = model(torch.tensor(X_te_s, dtype=torch.float32).to(device))

        y_prob_tr = torch.sigmoid(tr_logits).cpu().numpy()
        y_prob_te = torch.sigmoid(te_logits).cpu().numpy()

    return y_prob_tr, y_prob_te


# def run_lstm_windows(seq_train, seq_test):
#     """Run LSTM and BiLSTM for all four windows with per-student masking."""

#     # global max weeks per window (fixed sequence length)
#     quarter_config = {
#         "Q1":     ("q1_end", int(course_windows["q1_end"].max())),
#         "Q2":     ("q2_end", int(course_windows["q2_end"].max())),
#         "Q3":     ("q3_end", int(course_windows["q3_end"].max())),
#         "Master": ("q4_end", int(course_windows["q4_end"].max())),
#     }

#     lstm_rows = []
#     y_tr = seq_train[TARGET].astype(int).values
#     y_te = seq_test[TARGET].astype(int).values

#     for window_label, (q_end_col, global_max_wk) in quarter_config.items():
#         print(f"\n{'='*55}")
#         print(f"  {window_label} — LSTM/BiLSTM (up to week {global_max_wk}, "
#               f"masked per student)")
#         print(f"{'='*55}")

#         X_tr_seq = build_sequences_masked(seq_train, global_max_wk, q_end_col)
#         X_te_seq = build_sequences_masked(seq_test,  global_max_wk, q_end_col)

#         if X_tr_seq is None:
#             print(f"  No weekly columns found — skipping")
#             continue

#         print(f"  Train: {X_tr_seq.shape}  Test: {X_te_seq.shape}")

#         for name, bidir in [("LSTM", False), ("BiLSTM", True)]:
#             print(f"\n  Training {name}...")
#             y_prob, y_pred = train_seq_model(
#                 X_tr_seq, y_tr, X_te_seq, y_te,
#                 bidirectional=bidir
#             )
#             row = {
#                 "Window":    window_label,
#                 "Model":     name,
#                 "Accuracy":  round(accuracy_score(y_te, y_pred),                  4),
#                 "Precision": round(precision_score(y_te, y_pred, zero_division=0), 4),
#                 "Recall":    round(recall_score(y_te, y_pred, zero_division=0),    4),
#                 "F1":        round(f1_score(y_te, y_pred, zero_division=0),        4),
#                 "ROC-AUC":   round(roc_auc_score(y_te, y_prob),                   4),
#                 "_model":    None,
#                 "_X_te":     X_te_seq,
#                 "_y_te":     y_te,
#                 "_features": ["content", "assessment", "social"],
#                 "_y_prob":   y_prob,
#             }
#             lstm_rows.append(row)
#             print(f"  {name:22s}  "
#                   f"Acc={row['Accuracy']:.3f}  "
#                   f"Prec={row['Precision']:.3f}  "
#                   f"Rec={row['Recall']:.3f}  "
#                   f"F1={row['F1']:.3f}  "
#                   f"AUC={row['ROC-AUC']:.3f}")

#     return pd.DataFrame(lstm_rows)


# print("LSTM + BiLSTM defined.")
def run_lstm_windows(seq_train, seq_test):
    """
    Run LSTM and BiLSTM for all windows with:
      - per-student masking
      - validation-based early stopping
      - threshold tuning on train only
    """
    quarter_config = {
        "Q1": ("q1_end", int(course_windows["q1_end"].max())),
        "Q2": ("q2_end", int(course_windows["q2_end"].max())),
        "Q3": ("q3_end", int(course_windows["q3_end"].max())),
        "Master": ("q4_end", int(course_windows["q4_end"].max())),
    }

    lstm_rows = []
    y_tr = seq_train[TARGET].astype(int).values
    y_te = seq_test[TARGET].astype(int).values

    for window_label, (q_end_col, global_max_wk) in quarter_config.items():
        print(f"\n{'='*55}")
        print(f"  {window_label} — LSTM/BiLSTM (up to week {global_max_wk}, masked per student)")
        print(f"{'='*55}")

        X_tr_seq = build_sequences_masked(seq_train, global_max_wk, q_end_col)
        X_te_seq = build_sequences_masked(seq_test, global_max_wk, q_end_col)

        if X_tr_seq is None:
            print("  No weekly columns found — skipping")
            continue

        print(f"  Train: {X_tr_seq.shape}  Test: {X_te_seq.shape}")

        for name, bidir in [("LSTM", False), ("BiLSTM", True)]:
            print(f"\n  Training {name}...")
            y_prob_tr, y_prob_te = train_seq_model(
                X_tr_seq, y_tr, X_te_seq, y_te,
                bidirectional=bidir
            )

            best_thr, best_train_score = find_best_threshold(
                y_tr, y_prob_tr, metric="f1"
            )
            print(f"  Best train threshold for {name}: {best_thr:.2f} (train F1={best_train_score:.3f})")

            row = score_probabilities(
                name=name,
                y_te=y_te,
                y_prob_te=y_prob_te,
                window_label=window_label,
                features=["content", "assessment", "social"],
                model_obj=None,
                X_te=X_te_seq,
                threshold=best_thr
            )
            lstm_rows.append(row)

    return pd.DataFrame(lstm_rows)

In [ ]:
# Verify q_end columns exist in seq tables
for col in ["q1_end", "q2_end", "q3_end", "q4_end"]:
    if col not in ml_seq_train.columns:
        print(f"WARNING: {col} missing from ml_seq_train — merging now")
        ml_seq_train = ml_seq_train.merge(
            course_windows[["code_module","code_presentation",
                            "q1_end","q2_end","q3_end","q4_end"]],
            on=["code_module","code_presentation"], how="left")
        ml_seq_test = ml_seq_test.merge(
            course_windows[["code_module","code_presentation",
                            "q1_end","q2_end","q3_end","q4_end"]],
            on=["code_module","code_presentation"], how="left")
        break

print("q_end cols in ml_seq_train:", 
      [c for c in ml_seq_train.columns if "q" in c and "end" in c])

In [ ]:
# =============================================================================
# CELL 8 — Main Training Loop
# =============================================================================

QUARTER_MAX_WEEKS = {
    "Q1":     int(course_windows["q1_end"].median()),
    "Q2":     int(course_windows["q2_end"].median()),
    "Q3":     int(course_windows["q3_end"].median()),
    "Master": int(course_windows["q4_end"].max()),
}
print("Quarter max weeks:", QUARTER_MAX_WEEKS)

# all_cv_rows   = []
# all_test_rows = []

# for window_label, w in WINDOWS.items():
#     print(f"\n{'='*65}")
#     print(f"  {window_label}")
#     print(f"{'='*65}")

#     features = SELECTED_FEATURES[window_label]
#     train_df = w["train"]
#     test_df  = w["test"]

#     # Extract arrays
#     X_tr = train_df[features].fillna(0).values
#     X_te = test_df[features].fillna(0).values
#     y_tr = train_df[TARGET].astype(int).values
#     y_te = test_df[TARGET].astype(int).values

#     spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
#     print(f"\n    Train: {len(y_tr):,}  Test: {len(y_te):,}  "
#           f"SPW: {spw:.2f}  Features: {len(features)}")

#     # ── Cross validation (train only) ────────────────────────────────────────────
#     # Groups = presentation label per student for GroupKFold
#     # groups = train_df["code_presentation"].values
#     groups = (train_df["code_module"].astype(str) + "_" +
#     train_df["code_presentation"].astype(str)).values
#     cv_df  = run_cross_validation(window_label, X_tr, y_tr, spw, groups=groups)
#     all_cv_rows.append(cv_df)

#     # ── Final test evaluation ─────────────────────────────────────────────────
#     print(f"\n    --- Final Test (latest presentation per module) ---")
#     models = make_models(spw=spw)
#     for name, model in models.items():
#         model.fit(X_tr, y_tr)
#         row = score_row(name, model, X_te, y_te, window_label, features)
#         all_test_rows.append(pd.DataFrame([row]))

#     # ── MLP ─────────────────────────────────────────────────────────────
#     print(f"\n    Training MLP..")
#     y_prob_mlp, y_pred_mlp = train_mlp(X_tr, y_tr, X_te, y_te)
#     mlp_row = {
#         "Window":    window_label,
#         "Model":     "MLP",
#         "Accuracy":  round(accuracy_score(y_te, y_pred_mlp),                  4),
#         "Precision": round(precision_score(y_te, y_pred_mlp, zero_division=0), 4),
#         "Recall":    round(recall_score(y_te, y_pred_mlp, zero_division=0),    4),
#         "F1":        round(f1_score(y_te, y_pred_mlp, zero_division=0),        4),
#         "ROC-AUC":   round(roc_auc_score(y_te, y_prob_mlp),                   4),
#         "_model":    None,
#         "_X_te":     X_te,
#         "_y_te":     y_te,
#         "_features": features,
#         "_y_prob":   y_prob_mlp,
#     }
#     print(f"    {'MLP':22s}  "
#           f"Acc={mlp_row['Accuracy']:.3f}  "
#           f"Prec={mlp_row['Precision']:.3f}  "
#           f"Rec={mlp_row['Recall']:.3f}  "
#           f"F1={mlp_row['F1']:.3f}  "
#           f"AUC={mlp_row['ROC-AUC']:.3f}")
#     all_test_rows.append(pd.DataFrame([mlp_row]))

# # ── end of loop ───────────────────────────────────────────────────────────────
# results_cv   = pd.concat(all_cv_rows,   ignore_index=True)
# results_test = pd.concat(all_test_rows, ignore_index=True)

# display_cols = ["Window", "Model", "Accuracy",
#                 "Precision", "Recall", "F1", "ROC-AUC"]
# cv_display   = ["Window", "Model", "CV AUC", "CV AUC Std",
#                 "CV F1", "CV F1 Std", "CV Recall",
#                 "Train AUC", "Overfit Gap"]

# print("\n" + "="*95)
# print("CROSS VALIDATION RESULTS (train cohort only — 5-fold)")
# print("="*95)
# print(results_cv[cv_display].to_string(index=False))

# print("\n" + "="*95)
# print("FINAL TEST RESULTS (latest presentation per module)")
# print("="*95)
# print(results_test[display_cols].to_string(index=False))

# results_cv[cv_display].to_csv("results_cv.csv",   index=False)
# results_test[display_cols].to_csv("results_test.csv", index=False)
# print("\nSaved results_cv.csv and results_test.csv")

# print("\nOverfitting Check — CV Train vs CV Test AUC")
# print("="*65)
# for _, row in results_cv.iterrows():
#     gap    = row["Overfit Gap"]
#     status = "OK" if gap < 0.05 else "OVERFIT"
#     print(f"  {row['Window']:10s}  {row['Model']:22s}  "
#           f"Train={row['Train AUC']:.3f}  "
#           f"CV={row['CV AUC']:.3f}  "
#           f"Gap={gap:.3f}  {status}")

# print("\n\nAll windows complete.")
# =============================================================================
# CELL 8 — Main Training Loop (fixed)
# =============================================================================
from sklearn.metrics import f1_score, precision_score, recall_score

def find_best_threshold(y_true, y_prob, metric="f1"):
    """
    Select threshold on train/CV only.
    Default metric = F1.
    """
    thresholds = np.arange(0.10, 0.91, 0.02)
    best_thr = 0.50
    best_score = -1

    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)

        if metric == "f1":
            score = f1_score(y_true, y_pred, zero_division=0)
        elif metric == "recall":
            score = recall_score(y_true, y_pred, zero_division=0)
        elif metric == "precision":
            score = precision_score(y_true, y_pred, zero_division=0)
        else:
            raise ValueError("metric must be one of: f1, recall, precision")

        if score > best_score:
            best_score = score
            best_thr = thr

    return best_thr, best_score


def fit_model_and_get_probs(model, X_tr, y_tr, X_te):
    model.fit(X_tr, y_tr)
    y_prob_tr = model.predict_proba(X_tr)[:, 1]
    y_prob_te = model.predict_proba(X_te)[:, 1]
    return model, y_prob_tr, y_prob_te


def score_probabilities(name, y_te, y_prob_te, window_label, features,
                        model_obj=None, X_te=None, threshold=0.5):
    y_pred = (y_prob_te >= threshold).astype(int)

    row = {
        "Window": window_label,
        "Model": name,
        "Threshold": round(float(threshold), 3),
        "Accuracy": round(accuracy_score(y_te, y_pred), 4),
        "Precision": round(precision_score(y_te, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_te, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_te, y_pred, zero_division=0), 4),
        "ROC-AUC": round(roc_auc_score(y_te, y_prob_te), 4),
        "_model": model_obj,
        "_X_te": X_te,
        "_y_te": y_te,
        "_features": features,
        "_y_prob": y_prob_te,
    }

    print(
        f"    {name:22s}  "
        f"Thr={row['Threshold']:.2f}  "
        f"Acc={row['Accuracy']:.3f}  "
        f"Prec={row['Precision']:.3f}  "
        f"Rec={row['Recall']:.3f}  "
        f"F1={row['F1']:.3f}  "
        f"AUC={row['ROC-AUC']:.3f}"
    )
    return row

all_cv_rows = []
all_test_rows = []

for window_label, w in WINDOWS.items():
    print(f"\n{'='*65}")
    print(f"  {window_label}")
    print(f"{'='*65}")

    features = SELECTED_FEATURES[window_label]
    train_df = w["train"]
    test_df = w["test"]

    X_tr = train_df[features].fillna(0).values
    X_te = test_df[features].fillna(0).values
    y_tr = train_df[TARGET].astype(int).values
    y_te = test_df[TARGET].astype(int).values

    spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    print(f"\n    Train: {len(y_tr):,}  Test: {len(y_te):,}  SPW: {spw:.2f}  Features: {len(features)}")

    # Proper CV groups: module-presentation
    groups = (
        train_df["code_module"].astype(str) + "_" +
        train_df["code_presentation"].astype(str)
    ).values

    cv_df = run_cross_validation(window_label, X_tr, y_tr, spw, groups=groups)
    all_cv_rows.append(cv_df)

    print(f"\n    --- Final Test (latest presentation per module) ---")
    models = make_models(spw=spw)

    for name, model in models.items():
        model, y_prob_tr, y_prob_te = fit_model_and_get_probs(model, X_tr, y_tr, X_te)

        best_thr, best_train_score = find_best_threshold(
            y_tr, y_prob_tr, metric="f1"
        )
        print(f"    Best train threshold for {name}: {best_thr:.2f} (train F1={best_train_score:.3f})")

        row = score_probabilities(
            name=name,
            y_te=y_te,
            y_prob_te=y_prob_te,
            window_label=window_label,
            features=features,
            model_obj=model,
            X_te=X_te,
            threshold=best_thr
        )
        all_test_rows.append(pd.DataFrame([row]))

    # MLP
    print(f"\n    Training MLP...")
    y_prob_tr_mlp, y_prob_te_mlp = train_mlp(X_tr, y_tr, X_te, y_te)

    best_thr_mlp, best_train_score_mlp = find_best_threshold(
        y_tr, y_prob_tr_mlp, metric="f1"
    )
    print(f"    Best train threshold for MLP: {best_thr_mlp:.2f} (train F1={best_train_score_mlp:.3f})")

    mlp_row = score_probabilities(
        name="MLP",
        y_te=y_te,
        y_prob_te=y_prob_te_mlp,
        window_label=window_label,
        features=features,
        model_obj=None,
        X_te=X_te,
        threshold=best_thr_mlp
    )
    all_test_rows.append(pd.DataFrame([mlp_row]))

results_cv = pd.concat(all_cv_rows, ignore_index=True)
results_test = pd.concat(all_test_rows, ignore_index=True)

display_cols = ["Window", "Model", "Threshold", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
cv_display = ["Window", "Model", "CV AUC", "CV AUC Std", "CV F1", "CV F1 Std", "CV Recall", "Train AUC", "Overfit Gap"]

print("\n" + "="*95)
print("CROSS VALIDATION RESULTS (train cohort only)")
print("="*95)
print(results_cv[cv_display].to_string(index=False))

print("\n" + "="*95)
print("FINAL TEST RESULTS (latest presentation per module)")
print("="*95)
print(results_test[display_cols].to_string(index=False))

results_cv[cv_display].to_csv("results_cv.csv", index=False)
results_test[display_cols].to_csv("results_test.csv", index=False)

print("\nSaved results_cv.csv and results_test.csv")

print("\nOverfitting Check — CV Train vs CV Test AUC")
print("="*65)
for _, row in results_cv.iterrows():
    gap = row["Overfit Gap"]
    status = "OK" if gap < 0.05 else "OVERFIT"
    print(
        f"  {row['Window']:10s}  {row['Model']:22s}  "
        f"Train={row['Train AUC']:.3f}  "
        f"CV={row['CV AUC']:.3f}  "
        f"Gap={gap:.3f}  {status}"
    )

print("\nAll tabular windows complete.")

In [ ]:
# =============================================================================
# CELL 6b — HybridLSTM (sequence + static features)
# Add this cell AFTER the LSTM cell, BEFORE the results summary cell
# =============================================================================

# ── Step 1: Define which static features to use ──────────────────────────────
HYBRID_STATIC_COLS = [
    # Demographics
    "gender_bin", "disability_bin",
    "age_band_ord", "age_band_missing_flag",
    "highest_education_ord", "highest_education_missing_flag",
    "imd_ordinal", "imd_unknown_flag",
    "nation_England", "nation_Scotland", "nation_Wales", "nation_Northern_Ireland",
    "late_registration_flag",
    # Academic
    "studied_credits", "course_type_stem", "num_of_prev_attempts",
]
# 16 features total — matches static_input_size=16

# ── Step 2: Merge static features into seq tables ────────────────────────────
# ml_seq_train only has keys + target + weekly cols
# We need to add the static demographic/academic columns from ml_Q1_train
KEYS = ["id_student", "code_module", "code_presentation"]

_static_train = ml_Q1_train[KEYS + HYBRID_STATIC_COLS].drop_duplicates(subset=KEYS)
_static_test  = ml_Q1_test[KEYS + HYBRID_STATIC_COLS].drop_duplicates(subset=KEYS)

ml_seq_train_h = ml_seq_train.merge(_static_train, on=KEYS, how="left")
ml_seq_test_h  = ml_seq_test.merge(_static_test,   on=KEYS, how="left")

ml_seq_train_h[HYBRID_STATIC_COLS] = ml_seq_train_h[HYBRID_STATIC_COLS].fillna(0)
ml_seq_test_h[HYBRID_STATIC_COLS]  = ml_seq_test_h[HYBRID_STATIC_COLS].fillna(0)

print(f"ml_seq_train_h: {ml_seq_train_h.shape}")
print(f"ml_seq_test_h:  {ml_seq_test_h.shape}")


# ── Step 3: HybridLSTM model class ───────────────────────────────────────────
# class HybridLSTM(nn.Module):
#     def __init__(self, seq_input_size=3, static_input_size=16,
#                  hidden_size=64, num_layers=2, dropout=0.3,
#                  bidirectional=False):
#         super().__init__()                        # double underscores — fixed
#         self.bidirectional = bidirectional
#         self.lstm = nn.LSTM(
#             input_size=seq_input_size,
#             hidden_size=hidden_size,
#             num_layers=num_layers,
#             batch_first=True,
#             dropout=dropout if num_layers > 1 else 0.0,
#             bidirectional=bidirectional
#         )
#         factor = 2 if bidirectional else 1
#         self.static_net = nn.Sequential(
#             nn.Linear(static_input_size, 32),
#             nn.BatchNorm1d(32),
#             nn.ReLU(),
#             nn.Dropout(0.2)
#         )
#         self.classifier = nn.Sequential(
#             nn.Linear(hidden_size * factor + 32, 64),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(64, 1)
#             # No Sigmoid — BCEWithLogitsLoss expects raw logits
#         )

#     def forward(self, x_seq, x_static):
#         _, (h_n, _) = self.lstm(x_seq)
#         last = (torch.cat([h_n[-2], h_n[-1]], dim=1)
#                 if self.bidirectional else h_n[-1])
#         static_out = self.static_net(x_static)
#         combined   = torch.cat([last, static_out], dim=1)
#         return self.classifier(combined).squeeze(1)   # logits
# =============================================================================
# FIXED HybridLSTM
# =============================================================================
class HybridLSTM(nn.Module):
    def __init__(self, seq_input_size, static_input_size,
                 hidden_size=64, num_layers=2, dropout=0.3,
                 bidirectional=False):
        super().__init__()
        self.bidirectional = bidirectional

        self.lstm = nn.LSTM(
            input_size=seq_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )

        factor = 2 if bidirectional else 1

        self.static_net = nn.Sequential(
            nn.Linear(static_input_size, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * factor + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x_seq, x_static):
        _, (h_n, _) = self.lstm(x_seq)

        if self.bidirectional:
            last = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            last = h_n[-1]

        static_out = self.static_net(x_static)
        combined = torch.cat([last, static_out], dim=1)
        return self.classifier(combined).squeeze(1)

# ── Step 4: Training function ─────────────────────────────────────────────────
# def train_hybrid_seq_model(X_tr_seq, X_tr_static, y_tr,
#                             X_te_seq, X_te_static, y_te,
#                             bidirectional=False,
#                             epochs=60, batch_size=512,
#                             lr=1e-3, patience=10):
#     torch.manual_seed(42)
#     np.random.seed(42)

#     idx = np.arange(len(y_tr))
#     tr_idx, val_idx = train_test_split(
#         idx, test_size=0.15, random_state=42, stratify=y_tr)

#     # Sequence split
#     X_tr_seq_i  = X_tr_seq[tr_idx]
#     X_val_seq   = X_tr_seq[val_idx]

#     # Static split — scale from train inner only (no leakage)
#     X_tr_static_i      = X_tr_static[tr_idx]
#     X_val_static       = X_tr_static[val_idx]
#     static_scaler      = RobustScaler()
#     X_tr_static_s      = static_scaler.fit_transform(X_tr_static_i)
#     X_val_static_s     = static_scaler.transform(X_val_static)
#     X_te_static_s      = static_scaler.transform(X_te_static)
#     X_tr_static_full_s = static_scaler.transform(X_tr_static)

#     # Sequence normalisation — from train inner only
#     mean            = X_tr_seq_i.mean(axis=(0, 1), keepdims=True)
#     std             = X_tr_seq_i.std(axis=(0, 1),  keepdims=True) + 1e-8
#     X_tr_seq_s      = (X_tr_seq_i  - mean) / std
#     X_val_seq_s     = (X_val_seq   - mean) / std
#     X_te_seq_s      = (X_te_seq    - mean) / std
#     X_tr_seq_full_s = (X_tr_seq    - mean) / std

#     y_tr_inner = y_tr[tr_idx]
#     y_val      = y_tr[val_idx]

#     pos_weight = torch.tensor(
#         [(y_tr_inner == 0).sum() / max((y_tr_inner == 1).sum(), 1)],
#         dtype=torch.float32).to(device)

#     train_loader = DataLoader(
#         TensorDataset(
#             torch.tensor(X_tr_seq_s,    dtype=torch.float32),
#             torch.tensor(X_tr_static_s, dtype=torch.float32),
#             torch.tensor(y_tr_inner,     dtype=torch.float32)
#         ),
#         batch_size=batch_size, shuffle=True, num_workers=0
#     )

#     model = HybridLSTM(
#         seq_input_size=3,
#         static_input_size=len(HYBRID_STATIC_COLS),
#         bidirectional=bidirectional
#     ).to(device)

#     criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
#     optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

#     best_state   = None
#     best_val_auc = -np.inf
#     wait         = 0

#     X_val_seq_t    = torch.tensor(X_val_seq_s,    dtype=torch.float32).to(device)
#     X_val_static_t = torch.tensor(X_val_static_s, dtype=torch.float32).to(device)

#     for epoch in range(epochs):
#         model.train()
#         epoch_loss = 0.0
#         for x_seq_b, x_static_b, yb in train_loader:
#             x_seq_b    = x_seq_b.to(device)
#             x_static_b = x_static_b.to(device)
#             yb         = yb.to(device)
#             optimizer.zero_grad()
#             logits = model(x_seq_b, x_static_b)
#             loss   = criterion(logits, yb)
#             loss.backward()
#             nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#             optimizer.step()
#             epoch_loss += loss.item()

#         model.eval()
#         with torch.no_grad():
#             val_logits = model(X_val_seq_t, X_val_static_t)
#             val_prob   = torch.sigmoid(val_logits).cpu().numpy()

#         val_auc = roc_auc_score(y_val, val_prob)

#         if val_auc > best_val_auc:
#             best_val_auc = val_auc
#             best_state   = {k: v.cpu().clone()
#                             for k, v in model.state_dict().items()}
#             wait = 0
#         else:
#             wait += 1

#         if (epoch + 1) % 20 == 0:
#             print(f"      Epoch {epoch+1}/{epochs}  "
#                   f"loss={epoch_loss/len(train_loader):.4f}  "
#                   f"val_auc={val_auc:.4f}")

#         if wait >= patience:
#             print(f"      Early stopping at epoch {epoch+1}")
#             break

#     model.load_state_dict(best_state)
#     model.eval()

#     with torch.no_grad():
#         tr_logits = model(
#             torch.tensor(X_tr_seq_full_s,    dtype=torch.float32).to(device),
#             torch.tensor(X_tr_static_full_s, dtype=torch.float32).to(device))
#         te_logits = model(
#             torch.tensor(X_te_seq_s,         dtype=torch.float32).to(device),
#             torch.tensor(X_te_static_s,      dtype=torch.float32).to(device))

#         y_prob_tr = torch.sigmoid(tr_logits).cpu().numpy()
#         y_prob_te = torch.sigmoid(te_logits).cpu().numpy()

#     return y_prob_tr, y_prob_te
# =============================================================================
# FIXED train_hybrid_seq_model
# =============================================================================
def train_hybrid_seq_model(X_tr_seq, X_tr_static, y_tr,
                           X_te_seq, X_te_static, y_te,
                           bidirectional=False,
                           epochs=60, batch_size=512,
                           lr=1e-3, patience=10):

    torch.manual_seed(42)
    np.random.seed(42)

    idx = np.arange(len(y_tr))
    tr_idx, val_idx = train_test_split(
        idx, test_size=0.15, random_state=42, stratify=y_tr
    )

    # split
    X_tr_seq_i = X_tr_seq[tr_idx]
    X_val_seq = X_tr_seq[val_idx]

    X_tr_static_i = X_tr_static[tr_idx]
    X_val_static = X_tr_static[val_idx]

    y_tr_inner = y_tr[tr_idx]
    y_val = y_tr[val_idx]

    # scale static from train-inner only
    static_scaler = RobustScaler()
    X_tr_static_s = static_scaler.fit_transform(X_tr_static_i)
    X_val_static_s = static_scaler.transform(X_val_static)
    X_te_static_s = static_scaler.transform(X_te_static)
    X_tr_static_full_s = static_scaler.transform(X_tr_static)

    # scale sequence from train-inner only
    mean = X_tr_seq_i.mean(axis=(0, 1), keepdims=True)
    std = X_tr_seq_i.std(axis=(0, 1), keepdims=True) + 1e-8

    X_tr_seq_s = (X_tr_seq_i - mean) / std
    X_val_seq_s = (X_val_seq - mean) / std
    X_te_seq_s = (X_te_seq - mean) / std
    X_tr_seq_full_s = (X_tr_seq - mean) / std

    pos_weight = torch.tensor(
        [(y_tr_inner == 0).sum() / max((y_tr_inner == 1).sum(), 1)],
        dtype=torch.float32
    ).to(device)

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_tr_seq_s, dtype=torch.float32),
            torch.tensor(X_tr_static_s, dtype=torch.float32),
            torch.tensor(y_tr_inner, dtype=torch.float32)
        ),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    # IMPORTANT: dynamic static_input_size
    model = HybridLSTM(
        seq_input_size=X_tr_seq.shape[2],
        static_input_size=X_tr_static.shape[1],
        bidirectional=bidirectional
    ).to(device)

    print(f"      Static input size used by model: {X_tr_static.shape[1]}")

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_state = None
    best_val_auc = -np.inf
    wait = 0

    X_val_seq_t = torch.tensor(X_val_seq_s, dtype=torch.float32).to(device)
    X_val_static_t = torch.tensor(X_val_static_s, dtype=torch.float32).to(device)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for x_seq_b, x_static_b, yb in train_loader:
            x_seq_b = x_seq_b.to(device)
            x_static_b = x_static_b.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(x_seq_b, x_static_b)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_seq_t, X_val_static_t)
            val_prob = torch.sigmoid(val_logits).cpu().numpy()

        val_auc = roc_auc_score(y_val, val_prob)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if (epoch + 1) % 20 == 0:
            print(f"      Epoch {epoch+1}/{epochs}  loss={epoch_loss/len(train_loader):.4f}  val_auc={val_auc:.4f}")

        if wait >= patience:
            print(f"      Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        tr_logits = model(
            torch.tensor(X_tr_seq_full_s, dtype=torch.float32).to(device),
            torch.tensor(X_tr_static_full_s, dtype=torch.float32).to(device)
        )
        te_logits = model(
            torch.tensor(X_te_seq_s, dtype=torch.float32).to(device),
            torch.tensor(X_te_static_s, dtype=torch.float32).to(device)
        )

        y_prob_tr = torch.sigmoid(tr_logits).cpu().numpy()
        y_prob_te = torch.sigmoid(te_logits).cpu().numpy()

    return y_prob_tr, y_prob_te

# ── Step 5: Window runner ─────────────────────────────────────────────────────
def run_hybrid_lstm_windows(seq_train_h, seq_test_h):
    """Run HybridLSTM and HybridBiLSTM for all four windows."""
    quarter_config = {
        "Q1":     ("q1_end", int(course_windows["q1_end"].max())),
        "Q2":     ("q2_end", int(course_windows["q2_end"].max())),
        "Q3":     ("q3_end", int(course_windows["q3_end"].max())),
        "Master": ("q4_end", int(course_windows["q4_end"].max())),
    }

    hybrid_rows = []
    y_tr = seq_train_h[TARGET].astype(int).values
    y_te = seq_test_h[TARGET].astype(int).values

    # Extract static arrays once — same for all windows
    X_tr_static = seq_train_h[HYBRID_STATIC_COLS].values.astype(np.float32)
    X_te_static = seq_test_h[HYBRID_STATIC_COLS].values.astype(np.float32)

    for window_label, (q_end_col, global_max_wk) in quarter_config.items():
        print(f"\n{'='*55}")
        print(f"  {window_label} — HybridLSTM (seq + static, up to week {global_max_wk})")
        print(f"{'='*55}")

        X_tr_seq = build_sequences_masked(seq_train_h, global_max_wk, q_end_col)
        X_te_seq = build_sequences_masked(seq_test_h,  global_max_wk, q_end_col)

        if X_tr_seq is None:
            print("  No weekly columns — skipping")
            continue

        print(f"  Seq shape:    train={X_tr_seq.shape}  test={X_te_seq.shape}")
        print(f"  Static shape: train={X_tr_static.shape}  test={X_te_static.shape}")

        for name, bidir in [("HybridLSTM", False), ("HybridBiLSTM", True)]:
            print(f"\n  Training {name}...")
            y_prob_tr, y_prob_te = train_hybrid_seq_model(
                X_tr_seq, X_tr_static, y_tr,
                X_te_seq, X_te_static, y_te,
                bidirectional=bidir
            )

            best_thr, best_train_f1 = find_best_threshold(
                y_tr, y_prob_tr, metric="f1"
            )
            print(f"  Best threshold: {best_thr:.2f}  train F1={best_train_f1:.3f}")

            row = score_probabilities(
                name=name,
                y_te=y_te,
                y_prob_te=y_prob_te,
                window_label=window_label,
                features=["seq+static"],
                model_obj=None,
                X_te=X_te_seq,
                threshold=best_thr
            )
            hybrid_rows.append(row)

    return pd.DataFrame(hybrid_rows)


print("HybridLSTM defined and ready.")

# # ── Step 6: Run it ────────────────────────────────────────────────────────────
# results_hybrid = run_hybrid_lstm_windows(ml_seq_train_h, ml_seq_test_h)
# print("\nHybrid results:")
# display_cols = ["Window", "Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
# print(results_hybrid[display_cols].to_string(index=False))

In [ ]:
import os

if os.path.exists("results_lstm.csv"):
    print("Loading saved LSTM results...")
    results_lstm = pd.read_csv("results_lstm.csv")
else:
    print("Training LSTM...")
    results_lstm = run_lstm_windows(ml_seq_train, ml_seq_test)
    results_lstm.to_csv("results_lstm.csv", index=False)
    print("Saved results_lstm.csv")

print(results_lstm[["Window","Model","Accuracy","F1","ROC-AUC"]].to_string(index=False))

In [ ]:
if os.path.exists("results_hybrid.csv"):
    print("Loading saved Hybrid results...")
    results_hybrid = pd.read_csv("results_hybrid.csv")
else:
    print("Training HybridLSTM...")
    results_hybrid = run_hybrid_lstm_windows(ml_seq_train_h, ml_seq_test_h)
    results_hybrid.to_csv("results_hybrid.csv", index=False)
    print("Saved results_hybrid.csv")

print(results_hybrid[["Window","Model","Accuracy","F1","ROC-AUC"]].to_string(index=False))

In [ ]:
results_test_full = pd.concat(
    [results_test, results_lstm, results_hybrid],
    ignore_index=True
)
display_cols = ["Window", "Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
print(results_test_full[display_cols].to_string(index=False))
results_test_full[display_cols].to_csv("results_test_full.csv", index=False)
print("Saved results_test_full.csv")

In [ ]:
# =============================================================================
# CELL 9 — Results Summary
# =============================================================================

display_cols = ["Window", "Model", "Accuracy",
                "Precision", "Recall", "F1", "ROC-AUC"]

cv_display   = ["Window", "Model", "CV AUC", "CV AUC Std",
                "CV F1", "CV F1 Std", "CV Recall",
                "Train AUC", "Overfit Gap"]

print("\n" + "="*95)
print("CROSS VALIDATION RESULTS (train cohort only — 5-fold)")
print("="*95)
print(results_cv[cv_display].to_string(index=False))

print("\n" + "="*95)
print("FINAL TEST RESULTS (latest presentation per module)")
print("="*95)
print(results_test[display_cols].to_string(index=False))

results_cv[cv_display].to_csv("results_cv.csv", index=False)
results_test[display_cols].to_csv("results_test.csv", index=False)
print("\nSaved results_cv.csv and results_test.csv")

In [ ]:
print(results_test_full["Model"].unique())
print(results_test_full.shape)

In [ ]:
# =============================================================================
# CELL 10 — Overfitting Check
# =============================================================================

print("Overfitting Check — CV Train vs CV Test AUC")
print("="*65)

for _, row in results_cv.iterrows():
    gap    = row["Overfit Gap"]
    status = "OK" if gap < 0.05 else "OVERFIT"
    print(f"  {row['Window']:10s}  {row['Model']:22s}  "
          f"Train={row['Train AUC']:.3f}  "
          f"CV={row['CV AUC']:.3f}  "
          f"Gap={gap:.3f}  {status}")

In [ ]:
# =============================================================================
# CELL 11 — Visualisations
# =============================================================================

window_order = ["Q1", "Q2", "Q3", "Master"]
MODEL_COLORS = {
    "Logistic Regression": "#1565C0",
    "SVM":                 "#6A1B9A",
    "Random Forest":       "#1B5E20",
    "XGBoost":             "#E65100",
    "MLP":                 "#F9A825",
    "LSTM":                "#C62828",
    "BiLSTM":              "#880E4F",
}

# Use the full results table with LSTM
results_plot = results_test_full

# ── Plot 1: Test performance across windows ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 7))
fig.suptitle("Model Performance — Cross-Presentation Deployment Protocol",
             fontsize=14, fontweight="bold")

for ax, metric in zip(axes, ["F1", "ROC-AUC"]):
    for model_name, color in MODEL_COLORS.items():
        subset = results_plot[
            results_plot["Model"] == model_name].copy()
        if subset.empty:
            continue
        subset = (subset.set_index("Window")
                        .reindex(window_order)
                        .reset_index())
        ls = "--" if model_name in [
            "MLP", "LSTM", "BiLSTM"] else "-"
        ax.plot(subset["Window"], subset[metric],
                marker="o", linewidth=2.2, markersize=8,
                color=color, linestyle=ls, label=model_name)
        # Replace the annotation loop with this
        for _, row in subset.iterrows():
            if pd.notna(row[metric]):
                # Only annotate Q3 and Master to avoid crowding
                if row["Window"] in ["Q3", "Master"]:
                    ax.annotate(
                        f"{row[metric]:.3f}",
                        (row["Window"], row[metric]),
                        textcoords="offset points",
                        xytext=(0, 8), ha="center",
                        fontsize=7, color=color)

    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_ylabel(metric)
    ax.set_xlabel("Training Window")
    ax.set_ylim(0.4, 1.0)
    ax.tick_params(axis="x", rotation=15)
    ax.legend(fontsize=9, loc="lower right")
    ax.axhline(0.5, color="grey", linewidth=0.8,
               linestyle=":", alpha=0.5)

plt.tight_layout()
plt.savefig("window_performance.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Plot 2: CV AUC vs Test AUC ────────────────────────────────────────────────
trad_models  = ["Logistic Regression", "SVM",
                "Random Forest", "XGBoost"]
cv_sub   = results_cv[results_cv["Model"].isin(trad_models)].copy()
test_sub = results_plot[results_plot["Model"].isin(trad_models)].copy()
merged   = cv_sub[["Window","Model","CV AUC"]].merge(
    test_sub[["Window","Model","ROC-AUC"]],
    on=["Window","Model"]
)

fig, ax = plt.subplots(figsize=(14, 6))
x, width = np.arange(len(merged)), 0.35
ax.bar(x - width/2, merged["CV AUC"],   width,
       label="CV AUC (train cohort)",
       color="#1565C0", edgecolor="white", alpha=0.85)
ax.bar(x + width/2, merged["ROC-AUC"], width,
       label="Test AUC (latest presentation)",
       color="#C62828", edgecolor="white", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(
    merged["Window"] + "\n" + merged["Model"].str.split().str[0],
    rotation=30, fontsize=8)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("ROC-AUC")
ax.set_title("CV vs Final Test AUC — Deployment Protocol",
             fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("cv_vs_test.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Plot 3: Heatmap ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Performance Heatmap — Window x Model",
             fontsize=14, fontweight="bold")

for ax, metric in zip(axes, ["F1", "ROC-AUC"]):
    pivot = (results_plot
             .pivot(index="Window", columns="Model", values=metric)
             .reindex(window_order))
    sns.heatmap(pivot, annot=True, fmt=".3f",
                cmap="RdYlGn", vmin=0.5, vmax=1.0,
                linewidths=0.5, ax=ax,
                annot_kws={"size": 10, "weight": "bold"})
    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=25)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig("heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Plot 4: Feature importance — RF per window ────────────────────────────────
# rf_res = results_plot[
#     (results_plot["Model"] == "Random Forest") &
#     (results_plot["model"].notna())
# ].reset_index(drop=True)

# if len(rf_res) > 0:
#     n   = len(rf_res)
#     fig, axes = plt.subplots(1, n, figsize=(6*n, 8))
#     if n == 1:
#         axes = [axes]
#     fig.suptitle("Feature Importance — Random Forest per Window",
#                  fontsize=13, fontweight="bold")
#     for ax, (_, row) in zip(axes, rf_res.iterrows()):
#         clf   = row["_model"].named_steps["clf"]
#         feats = row["_features"]
#         imps  = pd.Series(
#             clf.feature_importances_, index=feats).sort_values()
#         colors = ["#C62828" if v >= imps.quantile(0.7)
#                   else "#90A4AE" for v in imps.values]
#         imps.plot(kind="barh", ax=ax, color=colors,
#                   edgecolor="white", linewidth=0.4)
#         ax.set_title(row["Window"], fontsize=10, fontweight="bold")
#         ax.set_xlabel("Importance")
#         ax.tick_params(axis="y", labelsize=8)
#     plt.tight_layout()
#     plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
#     plt.show()
# ── Plot 4: Feature importance — RF per window ───────────────────────────────
rf_res = results_plot[
    (results_plot["Model"] == "Random Forest") &
    (results_plot["_model"].notna())
].reset_index(drop=True)

if len(rf_res) > 0:
    n = len(rf_res)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 8))
    if n == 1:
        axes = [axes]

    fig.suptitle("Feature Importance — Random Forest per Window",
                 fontsize=13, fontweight="bold")

    for ax, (_, row) in zip(axes, rf_res.iterrows()):
        clf = row["_model"].named_steps["clf"]
        feats = row["_features"]
        imps = pd.Series(clf.feature_importances_, index=feats).sort_values()

        colors = [
            "#C62828" if v >= imps.quantile(0.7) else "#90A4AE"
            for v in imps.values
        ]

        imps.plot(kind="barh", ax=ax, color=colors,
                  edgecolor="white", linewidth=0.4)

        ax.set_title(row["Window"], fontsize=10, fontweight="bold")
        ax.set_xlabel("Importance")
        ax.tick_params(axis="y", labelsize=8)

    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()


# ── Plot 5: Confusion matrices — RF ──────────────────────────────────────────
if len(rf_res) > 0:
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]

    fig.suptitle("Confusion Matrices — Random Forest",
                 fontsize=13, fontweight="bold")

    for ax, (_, row) in zip(axes, rf_res.iterrows()):
        thr = row["Threshold"] if "Threshold" in row else 0.5
        y_prob = row["_y_prob"]
        y_pred = (y_prob >= thr).astype(int)

        ConfusionMatrixDisplay.from_predictions(
            row["_y_te"],
            y_pred,
            display_labels=["Not At-Risk", "At-Risk"],
            cmap="Blues",
            ax=ax,
            colorbar=False
        )
        ax.set_title(row["Window"], fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
    plt.show()

# ── Plot 5: Confusion matrices — RF ──────────────────────────────────────────
if len(rf_res) > 0:
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1:
        axes = [axes]
    fig.suptitle("Confusion Matrices — Random Forest",
                 fontsize=13, fontweight="bold")
    for ax, (_, row) in zip(axes, rf_res.iterrows()):
        ConfusionMatrixDisplay.from_predictions(
            row["_y_te"],
            row["_model"].predict(row["_X_te"]),
            display_labels=["Not At-Risk", "At-Risk"],
            cmap="Blues", ax=ax, colorbar=False
        )
        ax.set_title(row["Window"], fontsize=9, fontweight="bold")
    plt.tight_layout()
    plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
    plt.show()

## Cluster-based ML models

In [ ]:
# -----------------------------------------------------------------------------
# SAVE BASELINE RESULTS
# -----------------------------------------------------------------------------
results_cv_baseline  = results_cv.copy()
results_test_baseline = results_test.copy()

def is_cluster_col(col):
    return (
        col.startswith("cluster_") or
        col.startswith("caf_")
    )

SELECTED_FEATURES_BASE = {
    k: [f for f in v if not is_cluster_col(f)]
    for k, v in SELECTED_FEATURES.items()
}

print("✓ Baseline results and selected features saved")
for w in ["Q1", "Q2", "Q3", "Master"]:
    print(f"{w}: {len(SELECTED_FEATURES_BASE[w])} features")

# -----------------------------------------------------------------------------
# MAKE CLUSTER-AUGMENTED COPIES (reuse cluster_models from above)
# cluster_master_id / cluster_Q1_id etc. already in ml_*_train from attach_clusters
# -----------------------------------------------------------------------------
ml_master_train_cl = ml_master_train.copy()
ml_master_test_cl  = ml_master_test.copy()
ml_Q1_train_cl     = ml_Q1_train.copy()
ml_Q1_test_cl      = ml_Q1_test.copy()
ml_Q2_train_cl     = ml_Q2_train.copy()
ml_Q2_test_cl      = ml_Q2_test.copy()
ml_Q3_train_cl     = ml_Q3_train.copy()
ml_Q3_test_cl      = ml_Q3_test.copy()
print("✓ Cluster experiment copies created")

# -----------------------------------------------------------------------------
# CONFIG FOR CLUSTER-AWARE FEATURES
# -----------------------------------------------------------------------------
TARGET = "target_at_risk"

Z_FEATURES_BY_WINDOW = {
    "Q1":     ["AP_Q1",   "norm_total_Q1",         "active_weeks_ratio_Q1"],
    "Q2":     ["AP_Q2",   "norm_total_Q2",         "active_weeks_ratio_Q2"],
    "Q3":     ["AP_Q3",   "norm_total_Q3",         "active_weeks_ratio_Q3"],
    "Master": ["AP_full", "total_engagement_norm", "active_weeks_ratio_norm"],
}

CLUSTER_ID_COLS = {
    "Q1":     "cluster_Q1_id",
    "Q2":     "cluster_Q2_id",
    "Q3":     "cluster_Q3_id",
    "Master": "cluster_master_id",
}

def _safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").fillna(0.0)

def run_tabular_experiment(windows_dict, selected_features_dict, label="baseline"):
    all_cv_rows = []
    all_test_rows = []

    for window_label, w in windows_dict.items():
        print(f"\n{'='*70}")
        print(f"{label.upper()} — {window_label}")
        print(f"{'='*70}")

        features = selected_features_dict[window_label]
        train_df = w["train"]
        test_df  = w["test"]

        X_tr = train_df[features].fillna(0).values
        X_te = test_df[features].fillna(0).values
        y_tr = train_df[TARGET].astype(int).values
        y_te = test_df[TARGET].astype(int).values

        spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
        print(f"\nTrain: {len(y_tr):,} | Test: {len(y_te):,} | SPW: {spw:.2f} | Features: {len(features)}")

        groups = (
            train_df["code_module"].astype(str) + "_" +
            train_df["code_presentation"].astype(str)
        ).values

        # CV
        cv_df = run_cross_validation(window_label, X_tr, y_tr, spw, groups=groups)
        cv_df["Experiment"] = label
        all_cv_rows.append(cv_df)

        # Classical models
        print(f"\n--- Final Test ({label}) ---")
        models = make_models(spw=spw)

        for name, model in models.items():
            model, y_prob_tr, y_prob_te = fit_model_and_get_probs(model, X_tr, y_tr, X_te)

            best_thr, best_train_score = find_best_threshold(
                y_tr, y_prob_tr, metric="f1"
            )

            print(f"Best train threshold for {name}: {best_thr:.2f} (train F1={best_train_score:.3f})")

            row = score_probabilities(
                name=name,
                y_te=y_te,
                y_prob_te=y_prob_te,
                window_label=window_label,
                features=features,
                model_obj=model,
                X_te=X_te,
                threshold=best_thr
            )
            row["Experiment"] = label
            all_test_rows.append(pd.DataFrame([row]))

        # MLP
        print(f"\nTraining MLP ({label})...")
        y_prob_tr_mlp, y_prob_te_mlp = train_mlp(X_tr, y_tr, X_te, y_te)

        best_thr_mlp, best_train_score_mlp = find_best_threshold(
            y_tr, y_prob_tr_mlp, metric="f1"
        )

        print(f"Best train threshold for MLP: {best_thr_mlp:.2f} (train F1={best_train_score_mlp:.3f})")

        mlp_row = score_probabilities(
            name="MLP",
            y_te=y_te,
            y_prob_te=y_prob_te_mlp,
            window_label=window_label,
            features=features,
            model_obj=None,
            X_te=X_te,
            threshold=best_thr_mlp
        )
        mlp_row["Experiment"] = label
        all_test_rows.append(pd.DataFrame([mlp_row]))

    results_cv_exp = pd.concat(all_cv_rows, ignore_index=True)
    results_test_exp = pd.concat(all_test_rows, ignore_index=True)

    return results_cv_exp, results_test_exp


# -----------------------------------------------------------------------------
# CLUSTER-AWARE FEATURE ENGINEERING
# -----------------------------------------------------------------------------
# def add_cluster_aware_features_final(train_df, test_df, cluster_id_col, window, z_features_by_window):
#     prefix     = f"caf_{window.lower()}"
#     z_features = z_features_by_window[window]

#     result_train = train_df.copy()
#     result_test  = test_df.copy()

#     global_stats = {}
#     for f in z_features:
#         vals  = _safe_numeric(train_df[f])
#         mean_f = vals.mean()
#         std_f  = vals.std()
#         if pd.isna(std_f) or std_f < 1e-6:
#             std_f = 1e-6
#         global_stats[f] = {"mean": mean_f, "std": std_f}

#     global_stds = np.array([global_stats[f]["std"] for f in z_features], dtype=float)

#     cluster_stats    = {}
#     cluster_centroids = {}
#     cluster_risk     = {}

#     for cid in sorted(train_df[cluster_id_col].dropna().unique()):
#         mask  = train_df[cluster_id_col] == cid
#         stats = {}
#         cluster_risk[cid] = train_df.loc[mask, TARGET].mean()
#         centroid_vals = []
#         for f in z_features:
#             vals   = _safe_numeric(train_df.loc[mask, f])
#             mean_f = vals.mean()
#             std_f  = vals.std()
#             if pd.isna(std_f) or std_f < 1e-6:
#                 std_f = 1e-6
#             stats[f] = {"mean": mean_f, "std": std_f}
#             centroid_vals.append(mean_f)
#         cluster_stats[cid]     = stats
#         cluster_centroids[cid] = np.array(centroid_vals, dtype=float)

#     worst_cid      = max(cluster_risk, key=cluster_risk.get)
#     worst_centroid = cluster_centroids[worst_cid]

#     def apply_stats(df):
#         df        = df.copy()
#         label_map = ["AP", "total", "active"]
#         for cid in df[cluster_id_col].dropna().unique():
#             mask      = df[cluster_id_col] == cid
#             cid_stats = cluster_stats.get(cid, None)
#             for f, lbl in zip(z_features, label_map):
#                 vals   = _safe_numeric(df.loc[mask, f])
#                 mean_f = cid_stats[f]["mean"] if cid_stats else global_stats[f]["mean"]
#                 std_f  = cid_stats[f]["std"]  if cid_stats else global_stats[f]["std"]
#                 z      = (vals - mean_f) / std_f
#                 df.loc[mask, f"{prefix}_z_{lbl}"] = z.clip(-3, 3)
#             student_vals = np.column_stack([
#                 _safe_numeric(df.loc[mask, f]).values for f in z_features
#             ])
#             dist = np.sqrt((((student_vals - worst_centroid) / global_stds) ** 2).sum(axis=1))
#             df.loc[mask, f"{prefix}_dist_to_worst"] = dist
#         return df

#     result_train = apply_stats(result_train)
#     result_test  = apply_stats(result_test)

#     print(f"[{window}] cluster-aware features added")
#     print(f"  Worst cluster (train only): {worst_cid}")
#     print(f"  Worst-cluster risk: {cluster_risk[worst_cid]:.4f}")
#     return result_train, result_test

def add_cluster_aware_features_final(train_df, test_df, cluster_id_col, window, z_features_by_window):
    """
    Final cluster-aware features:
      - risk_prior : train-only cluster failure rate
      - z_AP       : AP score vs cluster peers
      - z_total    : total engagement vs cluster peers
      - z_active   : consistency vs cluster peers

    All statistics computed on TRAIN only, then applied to train/test.
    """
    prefix = f"caf_{window.lower()}"
    z_features = z_features_by_window[window]

    result_train = train_df.copy()
    result_test = test_df.copy()

    def _safe_numeric(series):
        return pd.to_numeric(series, errors="coerce").fillna(0.0)

    # global fallback stats
    global_stats = {}
    for f in z_features:
        vals = _safe_numeric(train_df[f])
        mean_f = vals.mean()
        std_f = vals.std()
        if pd.isna(std_f) or std_f < 1e-6:
            std_f = 1e-6
        global_stats[f] = {"mean": mean_f, "std": std_f}

    global_risk = train_df[TARGET].mean()

    # cluster stats from TRAIN only
    cluster_stats = {}
    cluster_risk = {}

    for cid in sorted(train_df[cluster_id_col].dropna().unique()):
        mask = train_df[cluster_id_col] == cid
        stats = {}

        risk_val = train_df.loc[mask, TARGET].mean()
        cluster_risk[cid] = risk_val
        stats["risk_prior"] = risk_val

        for f in z_features:
            vals = _safe_numeric(train_df.loc[mask, f])
            mean_f = vals.mean()
            std_f = vals.std()
            if pd.isna(std_f) or std_f < 1e-6:
                std_f = 1e-6
            stats[f] = {"mean": mean_f, "std": std_f}

        cluster_stats[cid] = stats

    worst_cid = max(cluster_risk, key=cluster_risk.get)

    def apply_stats(df):
        df = df.copy()
        labels = ["AP", "total", "active"]

        # default fallback values
        df[f"{prefix}_risk_prior"] = global_risk
        for lbl in labels:
            df[f"{prefix}_z_{lbl}"] = 0.0

        for cid in df[cluster_id_col].dropna().unique():
            mask = df[cluster_id_col] == cid
            cid_stats = cluster_stats.get(cid, None)

            if cid_stats is None:
                risk_val = global_risk
            else:
                risk_val = cid_stats["risk_prior"]

            df.loc[mask, f"{prefix}_risk_prior"] = risk_val

            for f, lbl in zip(z_features, labels):
                vals = _safe_numeric(df.loc[mask, f])

                if cid_stats is None:
                    mean_f = global_stats[f]["mean"]
                    std_f = global_stats[f]["std"]
                else:
                    mean_f = cid_stats[f]["mean"]
                    std_f = cid_stats[f]["std"]

                z = (vals - mean_f) / std_f
                df.loc[mask, f"{prefix}_z_{lbl}"] = z.clip(-3, 3)

        return df

    result_train = apply_stats(result_train)
    result_test = apply_stats(result_test)

    print(f"[{window}] cluster-aware features added")
    print(f"  Worst cluster (train only): {worst_cid}")
    print(f"  Worst-cluster risk: {cluster_risk[worst_cid]:.4f}")

    return result_train, result_test

# # -----------------------------------------------------------------------------
# # BUILD CAF TABLES
# # -----------------------------------------------------------------------------
# ml_Q1_train_caf,     ml_Q1_test_caf     = add_cluster_aware_features_final(ml_Q1_train_cl,     ml_Q1_test_cl,     CLUSTER_ID_COLS["Q1"],     "Q1",     Z_FEATURES_BY_WINDOW)
# ml_Q2_train_caf,     ml_Q2_test_caf     = add_cluster_aware_features_final(ml_Q2_train_cl,     ml_Q2_test_cl,     CLUSTER_ID_COLS["Q2"],     "Q2",     Z_FEATURES_BY_WINDOW)
# ml_Q3_train_caf,     ml_Q3_test_caf     = add_cluster_aware_features_final(ml_Q3_train_cl,     ml_Q3_test_cl,     CLUSTER_ID_COLS["Q3"],     "Q3",     Z_FEATURES_BY_WINDOW)
# ml_master_train_caf, ml_master_test_caf = add_cluster_aware_features_final(ml_master_train_cl, ml_master_test_cl, CLUSTER_ID_COLS["Master"], "Master", Z_FEATURES_BY_WINDOW)

# print("\n✓ Final cluster-aware tables built")

# # -----------------------------------------------------------------------------
# # FEATURE LISTS + WINDOWS
# # -----------------------------------------------------------------------------
# CAF_COLS_BY_WINDOW = {
#     "Q1":     [c for c in ml_Q1_train_caf.columns     if c.startswith("caf_q1_")],
#     "Q2":     [c for c in ml_Q2_train_caf.columns     if c.startswith("caf_q2_")],
#     "Q3":     [c for c in ml_Q3_train_caf.columns     if c.startswith("caf_q3_")],
#     "Master": [c for c in ml_master_train_caf.columns if c.startswith("caf_master_")],
# }

# WINDOWS_CAF = {
#     "Q1":     {"train": ml_Q1_train_caf,     "test": ml_Q1_test_caf},
#     "Q2":     {"train": ml_Q2_train_caf,     "test": ml_Q2_test_caf},
#     "Q3":     {"train": ml_Q3_train_caf,     "test": ml_Q3_test_caf},
#     "Master": {"train": ml_master_train_caf, "test": ml_master_test_caf},
# }
# -----------------------------------------------------------------------------
# BUILD CAF TABLES
# -----------------------------------------------------------------------------
ml_Q1_train_caf, ml_Q1_test_caf = add_cluster_aware_features_final(
    ml_Q1_train_cl, ml_Q1_test_cl, CLUSTER_ID_COLS["Q1"], "Q1", Z_FEATURES_BY_WINDOW
)
ml_Q2_train_caf, ml_Q2_test_caf = add_cluster_aware_features_final(
    ml_Q2_train_cl, ml_Q2_test_cl, CLUSTER_ID_COLS["Q2"], "Q2", Z_FEATURES_BY_WINDOW
)
ml_Q3_train_caf, ml_Q3_test_caf = add_cluster_aware_features_final(
    ml_Q3_train_cl, ml_Q3_test_cl, CLUSTER_ID_COLS["Q3"], "Q3", Z_FEATURES_BY_WINDOW
)
ml_master_train_caf, ml_master_test_caf = add_cluster_aware_features_final(
    ml_master_train_cl, ml_master_test_cl, CLUSTER_ID_COLS["Master"], "Master", Z_FEATURES_BY_WINDOW
)

print("\n✓ Final cluster-aware tables built")

# -----------------------------------------------------------------------------
# FEATURE LISTS + WINDOWS
# -----------------------------------------------------------------------------
CAF_COLS_BY_WINDOW = {
    "Q1": [
        "caf_q1_risk_prior",
        "caf_q1_z_AP",
        "caf_q1_z_total",
        "caf_q1_z_active",
    ],
    "Q2": [
        "caf_q2_risk_prior",
        "caf_q2_z_AP",
        "caf_q2_z_total",
        "caf_q2_z_active",
    ],
    "Q3": [
        "caf_q3_risk_prior",
        "caf_q3_z_AP",
        "caf_q3_z_total",
        "caf_q3_z_active",
    ],
    "Master": [
        "caf_master_risk_prior",
        "caf_master_z_AP",
        "caf_master_z_total",
        "caf_master_z_active",
    ],
}

WINDOWS_CAF = {
    "Q1":     {"train": ml_Q1_train_caf,     "test": ml_Q1_test_caf},
    "Q2":     {"train": ml_Q2_train_caf,     "test": ml_Q2_test_caf},
    "Q3":     {"train": ml_Q3_train_caf,     "test": ml_Q3_test_caf},
    "Master": {"train": ml_master_train_caf, "test": ml_master_test_caf},
}

# SELECTED_FEATURES_CLUSTER_AWARE_FINAL = {
#     w: SELECTED_FEATURES_BASE[w] + CAF_COLS_BY_WINDOW[w]
#     for w in WINDOWS_CAF
# }
SELECTED_FEATURES_CLUSTER_AWARE_FINAL = {
    w: SELECTED_FEATURES_BASE[w] + CAF_COLS_BY_WINDOW[w]
    for w in WINDOWS_CAF
}

for w in ["Q1", "Q2", "Q3", "Master"]:
    print(f"{w}: baseline={len(SELECTED_FEATURES_BASE[w])} | cluster-aware extra={len(CAF_COLS_BY_WINDOW[w])} | total={len(SELECTED_FEATURES_CLUSTER_AWARE_FINAL[w])}")

# -----------------------------------------------------------------------------
# RUN EXPERIMENT
# -----------------------------------------------------------------------------
results_cv_caf, results_test_caf = run_tabular_experiment(
    WINDOWS_CAF,
    SELECTED_FEATURES_CLUSTER_AWARE_FINAL,
    label="cluster_aware_final"
)

print("\n✓ Final cluster-aware experiment complete")

In [ ]:
# -----------------------------------------------------------------------------
# SAVE BASELINE RESULTS
# -----------------------------------------------------------------------------
results_cv_baseline  = results_cv.copy()
results_test_baseline = results_test.copy()

def is_cluster_col(col):
    return (
        col.startswith("cluster_") or
        col.startswith("caf_")
    )

SELECTED_FEATURES_BASE = {
    k: [f for f in v if not is_cluster_col(f)]
    for k, v in SELECTED_FEATURES.items()
}

print("✓ Baseline results and selected features saved")
for w in ["Q1", "Q2", "Q3", "Master"]:
    print(f"{w}: {len(SELECTED_FEATURES_BASE[w])} features")

# -----------------------------------------------------------------------------
# MAKE CLUSTER-AUGMENTED COPIES (reuse cluster_models from above)
# cluster_master_id / cluster_Q1_id etc. already in ml_*_train from attach_clusters
# -----------------------------------------------------------------------------
ml_master_train_cl = ml_master_train.copy()
ml_master_test_cl  = ml_master_test.copy()
ml_Q1_train_cl     = ml_Q1_train.copy()
ml_Q1_test_cl      = ml_Q1_test.copy()
ml_Q2_train_cl     = ml_Q2_train.copy()
ml_Q2_test_cl      = ml_Q2_test.copy()
ml_Q3_train_cl     = ml_Q3_train.copy()
ml_Q3_test_cl      = ml_Q3_test.copy()
print("✓ Cluster experiment copies created")

# -----------------------------------------------------------------------------
# CONFIG FOR CLUSTER-AWARE FEATURES
# -----------------------------------------------------------------------------
TARGET = "target_at_risk"

Z_FEATURES_BY_WINDOW = {
    "Q1":     ["AP_Q1",   "norm_total_Q1",         "active_weeks_ratio_Q1"],
    "Q2":     ["AP_Q2",   "norm_total_Q2",         "active_weeks_ratio_Q2"],
    "Q3":     ["AP_Q3",   "norm_total_Q3",         "active_weeks_ratio_Q3"],
    "Master": ["AP_full", "total_engagement_norm", "active_weeks_ratio_norm"],
}

CLUSTER_ID_COLS = {
    "Q1":     "cluster_Q1_id",
    "Q2":     "cluster_Q2_id",
    "Q3":     "cluster_Q3_id",
    "Master": "cluster_master_id",
}

def _safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").fillna(0.0)

def run_tabular_experiment(windows_dict, selected_features_dict, label="baseline"):
    all_cv_rows = []
    all_test_rows = []

    for window_label, w in windows_dict.items():
        print(f"\n{'='*70}")
        print(f"{label.upper()} — {window_label}")
        print(f"{'='*70}")

        features = selected_features_dict[window_label]
        train_df = w["train"]
        test_df  = w["test"]

        X_tr = train_df[features].fillna(0).values
        X_te = test_df[features].fillna(0).values
        y_tr = train_df[TARGET].astype(int).values
        y_te = test_df[TARGET].astype(int).values

        spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
        print(f"\nTrain: {len(y_tr):,} | Test: {len(y_te):,} | SPW: {spw:.2f} | Features: {len(features)}")

        groups = (
            train_df["code_module"].astype(str) + "_" +
            train_df["code_presentation"].astype(str)
        ).values

        # CV
        cv_df = run_cross_validation(window_label, X_tr, y_tr, spw, groups=groups)
        cv_df["Experiment"] = label
        all_cv_rows.append(cv_df)

        # Classical models
        print(f"\n--- Final Test ({label}) ---")
        models = make_models(spw=spw)

        for name, model in models.items():
            model, y_prob_tr, y_prob_te = fit_model_and_get_probs(model, X_tr, y_tr, X_te)

            best_thr, best_train_score = find_best_threshold(
                y_tr, y_prob_tr, metric="f1"
            )

            print(f"Best train threshold for {name}: {best_thr:.2f} (train F1={best_train_score:.3f})")

            row = score_probabilities(
                name=name,
                y_te=y_te,
                y_prob_te=y_prob_te,
                window_label=window_label,
                features=features,
                model_obj=model,
                X_te=X_te,
                threshold=best_thr
            )
            row["Experiment"] = label
            all_test_rows.append(pd.DataFrame([row]))

        # MLP
        print(f"\nTraining MLP ({label})...")
        y_prob_tr_mlp, y_prob_te_mlp = train_mlp(X_tr, y_tr, X_te, y_te)

        best_thr_mlp, best_train_score_mlp = find_best_threshold(
            y_tr, y_prob_tr_mlp, metric="f1"
        )

        print(f"Best train threshold for MLP: {best_thr_mlp:.2f} (train F1={best_train_score_mlp:.3f})")

        mlp_row = score_probabilities(
            name="MLP",
            y_te=y_te,
            y_prob_te=y_prob_te_mlp,
            window_label=window_label,
            features=features,
            model_obj=None,
            X_te=X_te,
            threshold=best_thr_mlp
        )
        mlp_row["Experiment"] = label
        all_test_rows.append(pd.DataFrame([mlp_row]))

    results_cv_exp = pd.concat(all_cv_rows, ignore_index=True)
    results_test_exp = pd.concat(all_test_rows, ignore_index=True)

    return results_cv_exp, results_test_exp


# -----------------------------------------------------------------------------
# CLUSTER-AWARE FEATURE ENGINEERING
# -----------------------------------------------------------------------------
# def add_cluster_aware_features_final(train_df, test_df, cluster_id_col, window, z_features_by_window):
#     prefix     = f"caf_{window.lower()}"
#     z_features = z_features_by_window[window]

#     result_train = train_df.copy()
#     result_test  = test_df.copy()

#     global_stats = {}
#     for f in z_features:
#         vals  = _safe_numeric(train_df[f])
#         mean_f = vals.mean()
#         std_f  = vals.std()
#         if pd.isna(std_f) or std_f < 1e-6:
#             std_f = 1e-6
#         global_stats[f] = {"mean": mean_f, "std": std_f}

#     global_stds = np.array([global_stats[f]["std"] for f in z_features], dtype=float)

#     cluster_stats    = {}
#     cluster_centroids = {}
#     cluster_risk     = {}

#     for cid in sorted(train_df[cluster_id_col].dropna().unique()):
#         mask  = train_df[cluster_id_col] == cid
#         stats = {}
#         cluster_risk[cid] = train_df.loc[mask, TARGET].mean()
#         centroid_vals = []
#         for f in z_features:
#             vals   = _safe_numeric(train_df.loc[mask, f])
#             mean_f = vals.mean()
#             std_f  = vals.std()
#             if pd.isna(std_f) or std_f < 1e-6:
#                 std_f = 1e-6
#             stats[f] = {"mean": mean_f, "std": std_f}
#             centroid_vals.append(mean_f)
#         cluster_stats[cid]     = stats
#         cluster_centroids[cid] = np.array(centroid_vals, dtype=float)

#     worst_cid      = max(cluster_risk, key=cluster_risk.get)
#     worst_centroid = cluster_centroids[worst_cid]

#     def apply_stats(df):
#         df        = df.copy()
#         label_map = ["AP", "total", "active"]
#         for cid in df[cluster_id_col].dropna().unique():
#             mask      = df[cluster_id_col] == cid
#             cid_stats = cluster_stats.get(cid, None)
#             for f, lbl in zip(z_features, label_map):
#                 vals   = _safe_numeric(df.loc[mask, f])
#                 mean_f = cid_stats[f]["mean"] if cid_stats else global_stats[f]["mean"]
#                 std_f  = cid_stats[f]["std"]  if cid_stats else global_stats[f]["std"]
#                 z      = (vals - mean_f) / std_f
#                 df.loc[mask, f"{prefix}_z_{lbl}"] = z.clip(-3, 3)
#             student_vals = np.column_stack([
#                 _safe_numeric(df.loc[mask, f]).values for f in z_features
#             ])
#             dist = np.sqrt((((student_vals - worst_centroid) / global_stds) ** 2).sum(axis=1))
#             df.loc[mask, f"{prefix}_dist_to_worst"] = dist
#         return df

#     result_train = apply_stats(result_train)
#     result_test  = apply_stats(result_test)

#     print(f"[{window}] cluster-aware features added")
#     print(f"  Worst cluster (train only): {worst_cid}")
#     print(f"  Worst-cluster risk: {cluster_risk[worst_cid]:.4f}")
#     return result_train, result_test

def add_cluster_aware_features_final(train_df, test_df, cluster_id_col, window, z_features_by_window):
    """
    Final cluster-aware features:
      - risk_prior : train-only cluster failure rate
      - z_AP       : AP score vs cluster peers
      - z_total    : total engagement vs cluster peers
      - z_active   : consistency vs cluster peers

    All statistics computed on TRAIN only, then applied to train/test.
    """
    prefix = f"caf_{window.lower()}"
    z_features = z_features_by_window[window]

    result_train = train_df.copy()
    result_test = test_df.copy()

    def _safe_numeric(series):
        return pd.to_numeric(series, errors="coerce").fillna(0.0)

    # global fallback stats
    global_stats = {}
    for f in z_features:
        vals = _safe_numeric(train_df[f])
        mean_f = vals.mean()
        std_f = vals.std()
        if pd.isna(std_f) or std_f < 1e-6:
            std_f = 1e-6
        global_stats[f] = {"mean": mean_f, "std": std_f}

    global_risk = train_df[TARGET].mean()

    # cluster stats from TRAIN only
    cluster_stats = {}
    cluster_risk = {}

    for cid in sorted(train_df[cluster_id_col].dropna().unique()):
        mask = train_df[cluster_id_col] == cid
        stats = {}

        risk_val = train_df.loc[mask, TARGET].mean()
        cluster_risk[cid] = risk_val
        stats["risk_prior"] = risk_val

        for f in z_features:
            vals = _safe_numeric(train_df.loc[mask, f])
            mean_f = vals.mean()
            std_f = vals.std()
            if pd.isna(std_f) or std_f < 1e-6:
                std_f = 1e-6
            stats[f] = {"mean": mean_f, "std": std_f}

        cluster_stats[cid] = stats

    worst_cid = max(cluster_risk, key=cluster_risk.get)

    def apply_stats(df):
        df = df.copy()
        labels = ["AP", "total", "active"]

        # default fallback values
        df[f"{prefix}_risk_prior"] = global_risk
        for lbl in labels:
            df[f"{prefix}_z_{lbl}"] = 0.0

        for cid in df[cluster_id_col].dropna().unique():
            mask = df[cluster_id_col] == cid
            cid_stats = cluster_stats.get(cid, None)

            if cid_stats is None:
                risk_val = global_risk
            else:
                risk_val = cid_stats["risk_prior"]

            df.loc[mask, f"{prefix}_risk_prior"] = risk_val

            for f, lbl in zip(z_features, labels):
                vals = _safe_numeric(df.loc[mask, f])

                if cid_stats is None:
                    mean_f = global_stats[f]["mean"]
                    std_f = global_stats[f]["std"]
                else:
                    mean_f = cid_stats[f]["mean"]
                    std_f = cid_stats[f]["std"]

                z = (vals - mean_f) / std_f
                df.loc[mask, f"{prefix}_z_{lbl}"] = z.clip(-3, 3)

        return df

    result_train = apply_stats(result_train)
    result_test = apply_stats(result_test)

    print(f"[{window}] cluster-aware features added")
    print(f"  Worst cluster (train only): {worst_cid}")
    print(f"  Worst-cluster risk: {cluster_risk[worst_cid]:.4f}")

    return result_train, result_test

# # -----------------------------------------------------------------------------
# # BUILD CAF TABLES
# # -----------------------------------------------------------------------------
# ml_Q1_train_caf,     ml_Q1_test_caf     = add_cluster_aware_features_final(ml_Q1_train_cl,     ml_Q1_test_cl,     CLUSTER_ID_COLS["Q1"],     "Q1",     Z_FEATURES_BY_WINDOW)
# ml_Q2_train_caf,     ml_Q2_test_caf     = add_cluster_aware_features_final(ml_Q2_train_cl,     ml_Q2_test_cl,     CLUSTER_ID_COLS["Q2"],     "Q2",     Z_FEATURES_BY_WINDOW)
# ml_Q3_train_caf,     ml_Q3_test_caf     = add_cluster_aware_features_final(ml_Q3_train_cl,     ml_Q3_test_cl,     CLUSTER_ID_COLS["Q3"],     "Q3",     Z_FEATURES_BY_WINDOW)
# ml_master_train_caf, ml_master_test_caf = add_cluster_aware_features_final(ml_master_train_cl, ml_master_test_cl, CLUSTER_ID_COLS["Master"], "Master", Z_FEATURES_BY_WINDOW)

# print("\n✓ Final cluster-aware tables built")

# # -----------------------------------------------------------------------------
# # FEATURE LISTS + WINDOWS
# # -----------------------------------------------------------------------------
# CAF_COLS_BY_WINDOW = {
#     "Q1":     [c for c in ml_Q1_train_caf.columns     if c.startswith("caf_q1_")],
#     "Q2":     [c for c in ml_Q2_train_caf.columns     if c.startswith("caf_q2_")],
#     "Q3":     [c for c in ml_Q3_train_caf.columns     if c.startswith("caf_q3_")],
#     "Master": [c for c in ml_master_train_caf.columns if c.startswith("caf_master_")],
# }

# WINDOWS_CAF = {
#     "Q1":     {"train": ml_Q1_train_caf,     "test": ml_Q1_test_caf},
#     "Q2":     {"train": ml_Q2_train_caf,     "test": ml_Q2_test_caf},
#     "Q3":     {"train": ml_Q3_train_caf,     "test": ml_Q3_test_caf},
#     "Master": {"train": ml_master_train_caf, "test": ml_master_test_caf},
# }
# -----------------------------------------------------------------------------
# BUILD CAF TABLES
# -----------------------------------------------------------------------------
ml_Q1_train_caf, ml_Q1_test_caf = add_cluster_aware_features_final(
    ml_Q1_train_cl, ml_Q1_test_cl, CLUSTER_ID_COLS["Q1"], "Q1", Z_FEATURES_BY_WINDOW
)
ml_Q2_train_caf, ml_Q2_test_caf = add_cluster_aware_features_final(
    ml_Q2_train_cl, ml_Q2_test_cl, CLUSTER_ID_COLS["Q2"], "Q2", Z_FEATURES_BY_WINDOW
)
ml_Q3_train_caf, ml_Q3_test_caf = add_cluster_aware_features_final(
    ml_Q3_train_cl, ml_Q3_test_cl, CLUSTER_ID_COLS["Q3"], "Q3", Z_FEATURES_BY_WINDOW
)
ml_master_train_caf, ml_master_test_caf = add_cluster_aware_features_final(
    ml_master_train_cl, ml_master_test_cl, CLUSTER_ID_COLS["Master"], "Master", Z_FEATURES_BY_WINDOW
)

print("\n✓ Final cluster-aware tables built")

# -----------------------------------------------------------------------------
# FEATURE LISTS + WINDOWS
# -----------------------------------------------------------------------------
CAF_COLS_BY_WINDOW = {
    "Q1": [
        "caf_q1_risk_prior",
        "caf_q1_z_AP",
        "caf_q1_z_total",
        "caf_q1_z_active",
    ],
    "Q2": [
        "caf_q2_risk_prior",
        "caf_q2_z_AP",
        "caf_q2_z_total",
        "caf_q2_z_active",
    ],
    "Q3": [
        "caf_q3_risk_prior",
        "caf_q3_z_AP",
        "caf_q3_z_total",
        "caf_q3_z_active",
    ],
    "Master": [
        "caf_master_risk_prior",
        "caf_master_z_AP",
        "caf_master_z_total",
        "caf_master_z_active",
    ],
}

WINDOWS_CAF = {
    "Q1":     {"train": ml_Q1_train_caf,     "test": ml_Q1_test_caf},
    "Q2":     {"train": ml_Q2_train_caf,     "test": ml_Q2_test_caf},
    "Q3":     {"train": ml_Q3_train_caf,     "test": ml_Q3_test_caf},
    "Master": {"train": ml_master_train_caf, "test": ml_master_test_caf},
}

# SELECTED_FEATURES_CLUSTER_AWARE_FINAL = {
#     w: SELECTED_FEATURES_BASE[w] + CAF_COLS_BY_WINDOW[w]
#     for w in WINDOWS_CAF
# }
SELECTED_FEATURES_CLUSTER_AWARE_FINAL = {
    w: SELECTED_FEATURES_BASE[w] + CAF_COLS_BY_WINDOW[w]
    for w in WINDOWS_CAF
}

for w in ["Q1", "Q2", "Q3", "Master"]:
    print(f"{w}: baseline={len(SELECTED_FEATURES_BASE[w])} | cluster-aware extra={len(CAF_COLS_BY_WINDOW[w])} | total={len(SELECTED_FEATURES_CLUSTER_AWARE_FINAL[w])}")

# -----------------------------------------------------------------------------
# RUN EXPERIMENT
# -----------------------------------------------------------------------------
results_cv_caf, results_test_caf = run_tabular_experiment(
    WINDOWS_CAF,
    SELECTED_FEATURES_CLUSTER_AWARE_FINAL,
    label="cluster_aware_final"
)

print("\n✓ Final cluster-aware experiment complete")

In [ ]:
# =============================================================================
# CELL — HybridLSTM + Cluster-Aware Static Features
# Run this AFTER:
#   1) your baseline/CAF tables are built
#   2) HybridLSTM class + train_hybrid_seq_model() already exist
#   3) seq_train_h / seq_test_h are available
# =============================================================================

# -----------------------------------------------------------------------------
# STATIC FEATURES FOR HYBRID + CAF
# demographic + academic + cluster-aware only
# -----------------------------------------------------------------------------
HYBRID_STATIC_BASE = [
    # Demographics
    "gender_bin", "disability_bin",
    "age_band_ord", "age_band_missing_flag",
    "highest_education_ord", "highest_education_missing_flag",
    "imd_ordinal", "imd_unknown_flag",
    "nation_England", "nation_Scotland", "nation_Wales", "nation_Northern_Ireland",
    "late_registration_flag",

    # Academic
    "studied_credits", "course_type_stem", "num_of_prev_attempts",
]

HYBRID_STATIC_CAF = {
    "Q1": HYBRID_STATIC_BASE + [
        "caf_q1_risk_prior",
        "caf_q1_z_AP",
        "caf_q1_z_total",
        "caf_q1_z_active",
    ],
    "Q2": HYBRID_STATIC_BASE + [
        "caf_q2_risk_prior",
        "caf_q2_z_AP",
        "caf_q2_z_total",
        "caf_q2_z_active",
    ],
    "Q3": HYBRID_STATIC_BASE + [
        "caf_q3_risk_prior",
        "caf_q3_z_AP",
        "caf_q3_z_total",
        "caf_q3_z_active",
    ],
    "Master": HYBRID_STATIC_BASE + [
        "caf_master_risk_prior",
        "caf_master_z_AP",
        "caf_master_z_total",
        "caf_master_z_active",
    ],
}

# -----------------------------------------------------------------------------
# USE CAF TABLES AS STATIC SOURCE TABLES
# -----------------------------------------------------------------------------
WINDOW_TRAIN_TABLES_CAF = {
    "Q1": ml_Q1_train_caf,
    "Q2": ml_Q2_train_caf,
    "Q3": ml_Q3_train_caf,
    "Master": ml_master_train_caf,
}

WINDOW_TEST_TABLES_CAF = {
    "Q1": ml_Q1_test_caf,
    "Q2": ml_Q2_test_caf,
    "Q3": ml_Q3_test_caf,
    "Master": ml_master_test_caf,
}

# -----------------------------------------------------------------------------
# RUNNER
# -----------------------------------------------------------------------------
# def run_hybrid_lstm_caf_windows(seq_train_h, seq_test_h):
#     quarter_config = {
#         "Q1":     ("q1_end", int(course_windows["q1_end"].max())),
#         "Q2":     ("q2_end", int(course_windows["q2_end"].max())),
#         "Q3":     ("q3_end", int(course_windows["q3_end"].max())),
#         "Master": ("q4_end", int(course_windows["q4_end"].max())),
#     }

#     hybrid_rows = []
#     y_tr = seq_train_h[TARGET].astype(int).values
#     y_te = seq_test_h[TARGET].astype(int).values

#     for window_label, (q_end_col, global_max_wk) in quarter_config.items():
#         print(f"\n{'='*60}")
#         print(f"  {window_label} — HybridLSTM + CAF (seq + static)")
#         print(f"{'='*60}")

#         static_cols = [c for c in HYBRID_STATIC_CAF[window_label]
#                        if c in WINDOW_TRAIN_TABLES_CAF[window_label].columns]

#         print(f"  Static features ({len(static_cols)}): {static_cols}")

#         # one row per student from CAF train/test tables
#         train_static_df = (
#             WINDOW_TRAIN_TABLES_CAF[window_label][KEYS + static_cols]
#             .drop_duplicates(subset=KEYS)
#             .copy()
#         )
#         test_static_df = (
#             WINDOW_TEST_TABLES_CAF[window_label][KEYS + static_cols]
#             .drop_duplicates(subset=KEYS)
#             .copy()
#         )

#         # merge static features onto sequence rows
#         # seq_tr_w = seq_train_h.merge(train_static_df, on=KEYS, how="left")
#         # seq_te_w = seq_test_h.merge(test_static_df, on=KEYS, how="left")

#         # drop overlapping static cols first to avoid _x/_y suffixes
#         seq_train_base = seq_train_h.drop(columns=[c for c in static_cols if c in seq_train_h.columns], errors="ignore").copy()
#         seq_test_base  = seq_test_h.drop(columns=[c for c in static_cols if c in seq_test_h.columns], errors="ignore").copy()
#         seq_tr_w = seq_train_base.merge(train_static_df, on=KEYS, how="left")
#         seq_te_w = seq_test_base.merge(test_static_df, on=KEYS, how="left")

#         seq_tr_w[static_cols] = seq_tr_w[static_cols].fillna(0.0)
#         seq_te_w[static_cols] = seq_te_w[static_cols].fillna(0.0)

#         X_tr_static = seq_tr_w[static_cols].astype(np.float32).values
#         X_te_static = seq_te_w[static_cols].astype(np.float32).values

#         X_tr_seq = build_sequences_masked(seq_train_h, global_max_wk, q_end_col)
#         X_te_seq = build_sequences_masked(seq_test_h,  global_max_wk, q_end_col)

#         if X_tr_seq is None or X_te_seq is None:
#             print("  No sequence data found — skipping")
#             continue

#         print(f"  Seq shape:    train={X_tr_seq.shape}  test={X_te_seq.shape}")
#         print(f"  Static shape: train={X_tr_static.shape}  test={X_te_static.shape}")

#         for model_name, bidir in [("HybridLSTM_CAF", False), ("HybridBiLSTM_CAF", True)]:
#             print(f"\n  Training {model_name}...")

#             y_prob_tr, y_prob_te = train_hybrid_seq_model(
#                 X_tr_seq, X_tr_static, y_tr,
#                 X_te_seq, X_te_static, y_te,
#                 bidirectional=bidir
#             )

#             best_thr, best_train_f1 = find_best_threshold(
#                 y_tr, y_prob_tr, metric="f1"
#             )
#             print(f"  Best threshold: {best_thr:.2f}  train F1={best_train_f1:.3f}")

#             row = score_probabilities(
#                 name=model_name,
#                 y_te=y_te,
#                 y_prob_te=y_prob_te,
#                 window_label=window_label,
#                 features=static_cols,
#                 model_obj=None,
#                 X_te=X_te_seq,
#                 threshold=best_thr
#             )
#             row["Experiment"] = "hybrid_lstm_caf"
#             hybrid_rows.append(row)

#     return pd.DataFrame(hybrid_rows)
def run_hybrid_lstm_caf_windows(seq_train_h, seq_test_h):
    quarter_config = {
        "Q1":     ("q1_end", int(course_windows["q1_end"].max())),
        "Q2":     ("q2_end", int(course_windows["q2_end"].max())),
        "Q3":     ("q3_end", int(course_windows["q3_end"].max())),
        "Master": ("q4_end", int(course_windows["q4_end"].max())),
    }

    hybrid_rows = []
    y_tr = seq_train_h[TARGET].astype(int).values
    y_te = seq_test_h[TARGET].astype(int).values

    for window_label, (q_end_col, global_max_wk) in quarter_config.items():
        print(f"\n{'='*60}")
        print(f"  {window_label} — HybridLSTM + CAF (seq + static)")
        print(f"{'='*60}")

        static_cols = [c for c in HYBRID_STATIC_CAF[window_label]
                       if c in WINDOW_TRAIN_TABLES_CAF[window_label].columns]

        print(f"  Static features ({len(static_cols)}): {static_cols}")

        train_static_df = (
            WINDOW_TRAIN_TABLES_CAF[window_label][KEYS + static_cols]
            .drop_duplicates(subset=KEYS)
            .copy()
        )
        test_static_df = (
            WINDOW_TEST_TABLES_CAF[window_label][KEYS + static_cols]
            .drop_duplicates(subset=KEYS)
            .copy()
        )

        # IMPORTANT FIX: remove overlapping cols before merge
        seq_train_base = seq_train_h.drop(
            columns=[c for c in static_cols if c in seq_train_h.columns],
            errors="ignore"
        ).copy()

        seq_test_base = seq_test_h.drop(
            columns=[c for c in static_cols if c in seq_test_h.columns],
            errors="ignore"
        ).copy()

        seq_tr_w = seq_train_base.merge(train_static_df, on=KEYS, how="left")
        seq_te_w = seq_test_base.merge(test_static_df, on=KEYS, how="left")

        seq_tr_w[static_cols] = seq_tr_w[static_cols].fillna(0.0)
        seq_te_w[static_cols] = seq_te_w[static_cols].fillna(0.0)

        X_tr_static = seq_tr_w[static_cols].astype(np.float32).values
        X_te_static = seq_te_w[static_cols].astype(np.float32).values

        X_tr_seq = build_sequences_masked(seq_train_h, global_max_wk, q_end_col)
        X_te_seq = build_sequences_masked(seq_test_h,  global_max_wk, q_end_col)

        if X_tr_seq is None or X_te_seq is None:
            print("  No sequence data found — skipping")
            continue

        print(f"  Seq shape:    train={X_tr_seq.shape}  test={X_te_seq.shape}")
        print(f"  Static shape: train={X_tr_static.shape}  test={X_te_static.shape}")

        for model_name, bidir in [("HybridLSTM_CAF", False), ("HybridBiLSTM_CAF", True)]:
            print(f"\n  Training {model_name}...")

            y_prob_tr, y_prob_te = train_hybrid_seq_model(
                X_tr_seq, X_tr_static, y_tr,
                X_te_seq, X_te_static, y_te,
                bidirectional=bidir
            )

            best_thr, best_train_f1 = find_best_threshold(
                y_tr, y_prob_tr, metric="f1"
            )
            print(f"  Best threshold: {best_thr:.2f}  train F1={best_train_f1:.3f}")

            row = score_probabilities(
                name=model_name,
                y_te=y_te,
                y_prob_te=y_prob_te,
                window_label=window_label,
                features=static_cols,
                model_obj=None,
                X_te=X_te_seq,
                threshold=best_thr
            )
            row["Experiment"] = "hybrid_lstm_caf"
            hybrid_rows.append(row)

    return pd.DataFrame(hybrid_rows)

# -----------------------------------------------------------------------------
# RUN
# -----------------------------------------------------------------------------
# results_hybrid_caf = run_hybrid_lstm_caf_windows(seq_train_h, seq_test_h)
results_hybrid_caf = run_hybrid_lstm_caf_windows(ml_seq_train_h, ml_seq_test_h)

print("\nSaved HybridLSTM + CAF results")
display(results_hybrid_caf[["Window", "Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]])

In [ ]:
# =============================================================================
# COMBINED RESULTS TABLE — ALL EXPERIMENTS
# =============================================================================

# Standardise column names across all result dataframes
def standardise(df, experiment_label):
    df = df.copy()
    df["Experiment"] = experiment_label
    # rename ROC-AUC to AUC if needed
    if "ROC-AUC" in df.columns and "AUC" not in df.columns:
        df = df.rename(columns={"ROC-AUC": "AUC"})
    return df

# Collect all test result dataframes
all_results = pd.concat([
    standardise(results_test_baseline,  "Baseline"),
    standardise(results_test_caf,       "CA_Final"),
    standardise(results_hybrid_caf,     "HybridLSTM_CAF"),
], ignore_index=True)

# Keep only relevant columns
keep_cols = ["Experiment", "Window", "Model", "Accuracy", "Precision", "Recall", "F1", "AUC"]
# handle ROC-AUC name variant
if "ROC-AUC" in all_results.columns and "AUC" not in all_results.columns:
    all_results = all_results.rename(columns={"ROC-AUC": "AUC"})

all_results = all_results[[c for c in keep_cols if c in all_results.columns]]

# Round for readability
for col in ["Accuracy", "Precision", "Recall", "F1", "AUC"]:
    if col in all_results.columns:
        all_results[col] = all_results[col].round(4)

# Sort
window_order = {"Q1": 0, "Q2": 1, "Q3": 2, "Master": 3}
all_results["_w_order"] = all_results["Window"].map(window_order)
all_results = all_results.sort_values(
    ["_w_order", "Experiment", "Model"]
).drop(columns=["_w_order"]).reset_index(drop=True)

# Save
all_results.to_csv("results_all_experiments.csv", index=False)
print("✓ Saved results_all_experiments.csv")

# Display
print("\n" + "="*80)
print("COMBINED RESULTS — ALL EXPERIMENTS")
print("="*80)
print(all_results.to_string(index=False))

# Best per window
print("\n" + "="*80)
print("BEST MODEL PER WINDOW (by AUC)")
print("="*80)
best = all_results.loc[all_results.groupby("Window")["AUC"].idxmax()]
print(best[["Window", "Experiment", "Model", "AUC", "F1"]].to_string(index=False))

In [ ]:
print(results_lstm.columns.tolist())
print(results_lstm.head())

In [ ]:
# =============================================================================
# COMBINED RESULTS TABLE — ALL EXPERIMENTS
# =============================================================================

DISPLAY_COLS = ["Experiment", "Window", "Model", "Accuracy", "Precision", "Recall", "F1", "AUC"]

def standardise(df, experiment_label):
    df = df.copy()
    df["Experiment"] = experiment_label
    if "ROC-AUC" in df.columns:
        df = df.rename(columns={"ROC-AUC": "AUC"})
    # drop internal columns
    drop_cols = [c for c in df.columns if c.startswith("_")]
    df = df.drop(columns=drop_cols, errors="ignore")
    return df

all_results = pd.concat([
    standardise(results_test_baseline, "Baseline"),
    standardise(results_test_caf,      "CA_Final"),
    standardise(results_lstm,          "LSTM"),
    standardise(results_hybrid_caf,    "HybridLSTM_CAF"),
], ignore_index=True)

# Round
for col in ["Accuracy", "Precision", "Recall", "F1", "AUC"]:
    if col in all_results.columns:
        all_results[col] = all_results[col].round(4)

# Sort
window_order  = {"Q1": 0, "Q2": 1, "Q3": 2, "Master": 3}
all_results["_w"] = all_results["Window"].map(window_order)
exp_order = {"Baseline": 0, "CA_Final": 1, "LSTM": 2, "HybridLSTM_CAF": 3}
all_results["_e"] = all_results["Experiment"].map(exp_order)
all_results = all_results.sort_values(["_w", "_e", "Model"]).drop(
    columns=["_w", "_e"]
).reset_index(drop=True)

all_results = all_results[[c for c in DISPLAY_COLS if c in all_results.columns]]

# Save
all_results.to_csv("results_all_experiments.csv", index=False)
print("✓ Saved results_all_experiments.csv")

# Display full table
print("\n" + "="*85)
print("COMBINED RESULTS — ALL EXPERIMENTS")
print("="*85)
print(all_results.to_string(index=False))

# Best per window
print("\n" + "="*85)
print("BEST MODEL PER WINDOW (by AUC)")
print("="*85)
best = all_results.loc[all_results.groupby("Window")["AUC"].idxmax()]
print(best[["Window", "Experiment", "Model", "AUC", "F1"]].to_string(index=False))